# Exercises — AI for Power & Energy Systems

> **How to use this notebook.** Each task has a scaffold: `# TODO` markers, `____` blanks and function signatures with a docstring but no body. Fill them in.
>
> Most tasks end with a **self-check** cell that runs a few assertions. A passing check means the shapes and the obvious invariants are right; it does not mean the reasoning is. The analysis and reflection tasks have no check at all, because the thing being assessed is the argument.
>
> **Do not open `solution.ipynb` until you have attempted the corresponding exercise.**

Ten chapters, one per tutorial, 44 tasks in total.

Work them in order. Ideas carry forward — chapter 06 assumes you derived
attention in chapter 05, chapter 10 assumes you masked something in
chapter 04 — but *code* dependencies stay inside a chapter, so you can run
one chapter without having done the previous one. Any task that reuses an
earlier task's code says which, in its own prompt.

### Task types

| Type | What it asks for |
| --- | --- |
| **Coding Task** (30) | Implement something. The scaffold shows the shape; you write the body. |
| **Analysis Task** (8) | Run an experiment and interpret the numbers. Partly code, mostly judgement. |
| **Reflection Question** (6) | Write an argument. No code. Worth discussing with someone. |

### Difficulty

| | Meaning |
| --- | --- |
| ★ | Direct application of something the tutorial showed. |
| ★★ | Requires combining two ideas, or getting an index right. |
| ★★★ | Open-ended, or the correct answer is not the obvious one. |

### Chapters

| # | Topic | Builds on | Tasks | Hardest |
| --- | --- | --- | --- | --- |
| 01 | Classical Machine Learning | [`01_classical_machine_learning.ipynb`](01_classical_machine_learning.ipynb) | 5 | ★★ |
| 02 | Neural Networks | [`02_neural_networks.ipynb`](02_neural_networks.ipynb) | 4 | ★★ |
| 03 | RNNs and LSTMs | [`03_rnns_and_lstms.ipynb`](03_rnns_and_lstms.ipynb) | 4 | ★★★ |
| 04 | Representation Learning | [`04_representation_learning.ipynb`](04_representation_learning.ipynb) | 4 | ★★★ |
| 05 | Attention | [`05_attention.ipynb`](05_attention.ipynb) | 4 | ★★★ |
| 06 | Transformers | [`06_transformers.ipynb`](06_transformers.ipynb) | 4 | ★★★ |
| 07 | Language Models | [`07_language_models.ipynb`](07_language_models.ipynb) | 5 | ★★★ |
| 08 | Pretrained LLMs and Adaptation | [`08_pretrained_llms_and_adaptation.ipynb`](08_pretrained_llms_and_adaptation.ipynb) | 4 | ★★★ |
| 09 | Foundation Models Beyond LLMs | [`09_foundation_models_beyond_llms.ipynb`](09_foundation_models_beyond_llms.ipynb) | 4 | ★★★ |
| 10 | Towards Grid Foundation Models | [`10_grid_foundation_models.ipynb`](10_grid_foundation_models.ipynb) | 6 | ★★★ |

### Before you start

The environment is the course's own: `uv sync`, then select the
`ai-power-systems-course` kernel. Chapters 08 and 09 download pretrained
models the first time they run; set `AI_POWER_COURSE_OFFLINE=1` to skip
those cells and `AI_POWER_COURSE_FAST=1` to shrink every training budget.

In [ ]:
# Shared imports and the reproducibility seed. Every chapter below
# re-imports what it needs, so chapters can also be run in isolation.
import numpy as np
import pandas as pd
import torch

from ai_power_course.config import fast_mode, offline_mode, set_seed
from ai_power_course.plotting import use_course_style

use_course_style()
set_seed()

print(f"reduced (fast) configuration: {fast_mode()} | offline: {offline_mode()}")
print("Every result below is seeded. If a number differs from a neighbour's,")
print("it is because one of you changed something — which is the point.")

---

## Exercise 01 — Classical Machine Learning

*Builds on [`01_classical_machine_learning.ipynb`](01_classical_machine_learning.ipynb).*

Before any neural network, the discipline: framing the problem, splitting
the data so the split cannot lie to you, and building a baseline worth
beating.

The running problem for the whole exercise track is **day-ahead system load
forecasting** — predict the load 24 hours after the moment the forecast is
made. Every later chapter attacks the same problem with a different tool, so
the numbers are comparable throughout.

In [ ]:
from ai_power_course.config import set_seed
from ai_power_course.data import load_energy_data, make_supervised, time_split
from ai_power_course.metrics import mae, rmse

set_seed()
dataset = load_energy_data()
print(dataset.describe())

frame = dataset.frame
HORIZON = 24
train_raw, valid_raw, test_raw = time_split(frame)
print(f"\ntrain {len(train_raw):,} h | validation {len(valid_raw):,} h | "
      f"test {len(test_raw):,} h")

### Analysis Task 1.1 ★ — Inspect the dataset and find its structure

A forecasting model is a bet about structure. Before choosing one,
find out what structure is actually there. This dataset is hourly and
has three seasonalities and a weather driver; you should be able to
see all four.

**Your task.** Produce a two-panel figure: the mean load by hour of day (weekday
versus weekend on the same axes) and the mean load by month. Then
print the annual energy in TWh.

**Requirements**

- Use `frame`, which is already loaded.
- Both panels need axis labels and a title.
- Group with pandas rather than looping over rows.

<details><summary><b>Hints</b></summary>

- `frame.index` is a tz-aware `DatetimeIndex`, so `frame.index.hour` and `frame.index.month` are available directly.
- `frame.groupby([a, b]).load_mw.mean().unstack()` gives you a frame with one column per group.

</details>

**Expected outcome.** A double daily peak, a visibly lower weekend profile, and a winter
maximum. The annual energy should be on the order of a few hundred
TWh — this dataset is scaled to resemble Germany.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

# TODO 1: mean load by hour of day, split into weekday and weekend.
#         Hint: group by [frame.index.dayofweek >= 5, frame.index.hour]
hourly = ____
axes[0].plot(____, ____, label="weekday")
axes[0].plot(____, ____, label="weekend")
axes[0].set_xlabel(____)
axes[0].set_ylabel(____)
axes[0].set_title(____)
axes[0].legend()

# TODO 2: mean load by month
monthly = ____
axes[1].plot(monthly.index, monthly.to_numpy(), marker="o")
axes[1].set_xlabel(____)
axes[1].set_ylabel(____)
axes[1].set_title(____)

fig.tight_layout()
plt.show()

# TODO 3: annual energy in TWh (the data covers several whole years)
years = (frame.index[-1] - frame.index[0]).days / 365.25
annual_twh = ____
print(f"annual load energy: {annual_twh:.0f} TWh/a")

### Coding Task 1.2 ★ — Build a leakage-free temporal split

A random train/test split of a time series lets the model interpolate
between neighbouring hours it has already memorised. The score goes
up, the model does not get better, and nothing warns you.

Tutorial 01 measured the damage: a shuffled split flattered a
1-nearest-neighbour model by 30%.

**Your task.** Build the supervised tables for all three splits using
`make_supervised`, then verify by hand that no feature uses
information from after the forecast origin.

**Requirements**

- Produce `X_train, y_train, X_valid, y_valid, X_test, y_test`.
- Use `horizon=HORIZON` and pass `temperature_c` as an exogenous feature.
- The verification must compare against the raw series, not against another function from the same module.

<details><summary><b>Hints</b></summary>

- `make_supervised(part, horizon=..., exogenous=(...))` returns `(X, y)`.
- Row `t` is the forecast origin: `load_mw_lag0` is the load *at* `t`, and `y` is the load at `t + horizon`.

</details>

**Expected outcome.** Three aligned tables with no missing values, and three assertions that
pass. If an assertion fails, the features and the target are
misaligned — fix that before training anything.

In [ ]:
import numpy as np

EXOGENOUS = ("temperature_c",)

# TODO 1: build the three supervised tables
X_train, y_train = ____
X_valid, y_valid = ____
X_test, y_test = ____

print(f"X_train {X_train.shape} | X_valid {X_valid.shape} | X_test {X_test.shape}")

# TODO 2: verify the alignment against the raw series.
#         Pick one origin and check lag0, lag24 and the target.
probe = X_train.index[1000]
series = train_raw.load_mw
position = series.index.get_loc(probe)

assert np.isclose(X_train.loc[probe, "load_mw_lag0"], ____)
assert np.isclose(X_train.loc[probe, "load_mw_lag24"], ____)
assert np.isclose(y_train.loc[probe], ____)
print("alignment verified against the raw series")

In [ ]:
assert not X_train.isna().any().any(), "features contain NaN"
assert len(X_train) == len(y_train), "X and y are not aligned"
assert X_train.index.max() < X_test.index.min(), "the splits overlap in time"
print("Basic checks passed.")

### Coding Task 1.3 ★ — Persistence and seasonal-naive baselines

A forecast is only as good as what it beats, and "better than the
mean" is not an achievement on a series with a daily cycle.

Note an identity that catches people out: on hourly data at a
24-hour horizon, persistence and a 24-hour seasonal naive are the
*same* forecast.

*Reuses what you wrote in 1.2 — do those first.*

**Your task.** Implement a persistence forecast — the predicted load at the target
time equals the load observed at the forecast origin — and a weekly
seasonal naive. Score both.

**Requirements**

- Both must return a `pandas.Series` indexed by the forecast origins in `y_test.index`.
- Neither may use a value from after the origin.
- Report MAE and RMSE for each.

<details><summary><b>Hints</b></summary>

- `frame.load_mw.index.get_indexer(origins)` converts timestamps to integer positions you can offset.
- For a weekly naive at horizon h, the value you want sits `168 - h` steps before the origin.

</details>

**Expected outcome.** The weekly naive should clearly beat persistence: knowing the weekday
matters more at a 24-hour horizon than knowing the last value. Both
should be far better than predicting the training mean.

In [ ]:
import pandas as pd

history = frame.load_mw
origins = y_test.index
positions = history.index.get_indexer(origins)

def persistence_forecast(history, origins, horizon):
    """Predict y[t + horizon] as the value observed at t."""
    # TODO: return a Series indexed by `origins`
    raise NotImplementedError

def weekly_naive_forecast(history, origins, horizon, season=168):
    """Predict y[t + horizon] as y[t + horizon - season]."""
    # TODO: work out the offset from the origin, then index
    raise NotImplementedError

baseline = {
    "persistence": persistence_forecast(history, origins, HORIZON),
    "seasonal naive (168 h)": weekly_naive_forecast(history, origins, HORIZON),
    "training mean": pd.Series(y_train.mean(), index=origins),
}
for name, prediction in baseline.items():
    print(f"{name:24s} MAE {mae(y_test, prediction):8,.1f} MW   "
          f"RMSE {rmse(y_test, prediction):8,.1f} MW")

In [ ]:
for _name, _p in baseline.items():
    assert _p.index.equals(y_test.index), f"{_name} is not aligned to the origins"
    assert _p.notna().all(), f"{_name} has gaps"
print("Basic checks passed.")

### Coding Task 1.4 ★★ — Linear regression and gradient boosting

Two hypothesis classes on identical features. The linear model is
interpretable and cannot represent the V-shaped temperature response;
the tree ensemble can, and needs no feature scaling.

*Reuses what you wrote in 1.2, 1.3 — do those first.*

**Your task.** Fit a linear regression and a `HistGradientBoostingRegressor` on the
training split, then evaluate both on the test split alongside the
baselines from task 1.3.

**Requirements**

- Scale the features for the linear model with a `Pipeline`, so the scaler is fitted on training data only.
- Use `random_state=0` for the boosting model so the result is reproducible.
- Collect everything into one sorted results table.

<details><summary><b>Hints</b></summary>

- `make_pipeline(StandardScaler(), LinearRegression())` keeps the scaler inside the model, which is what makes cross-validation honest.
- `HistGradientBoostingRegressor(max_iter=..., learning_rate=...)` needs no scaling at all — trees compare, they do not add.

</details>

**Expected outcome.** Both learned models should beat every baseline comfortably. Gradient
boosting should beat the linear model, because the temperature
response and the hour x weekday interaction are exactly what trees
represent well.

In [ ]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

set_seed()

# TODO 1: a scaled linear model
linear = make_pipeline(____, ____)
linear.fit(____, ____)

# TODO 2: gradient boosting
boosting = HistGradientBoostingRegressor(
    max_iter=____,
    learning_rate=____,
    random_state=0,
)
boosting.fit(____, ____)

predictions = dict(baseline)
predictions["linear regression"] = ____
predictions["gradient boosting"] = ____

results = pd.DataFrame(
    {name: {"MAE": mae(y_test, p), "RMSE": rmse(y_test, p)}
     for name, p in predictions.items()}
).T.sort_values("MAE")
display(results.round(1))

In [ ]:
assert results.index[0] in {"gradient boosting", "linear regression"}, \
    "a learned model should top the table — check the feature alignment"
assert results.MAE.min() < results.loc["seasonal naive (168 h)", "MAE"], \
    "the learned models must beat the weekly naive baseline"
print("Basic checks passed.")

### Reflection Question 1.5 ★★ — Why does a good model sometimes lose to persistence?

In this notebook the learned models won. In published energy
forecasting results they frequently do not, and the M4 competition
found the same across thousands of series.

**Your task.** Give three distinct mechanisms by which a well-implemented machine
learning model can fail to beat persistence on a real forecasting
task. For each, say what you would measure to detect it.

**Expected outcome.** Good answers separate *problem* properties (the horizon is short, the
series is near-random-walk) from *method* failures (distribution
shift between train and test, a leak in the baseline's favour,
over-tuned hyper-parameters).

**Mechanism 1.** …

*How I would detect it:* …

**Mechanism 2.** …

*How I would detect it:* …

**Mechanism 3.** …

*How I would detect it:* …

---

## Exercise 02 — Neural Networks

*Builds on [`02_neural_networks.ipynb`](02_neural_networks.ipynb).*

The same forecasting problem, with the non-linearity learned instead of
engineered. You will write a neuron and a gradient step by hand before
touching `autograd`, because `loss.backward()` should be a convenience, not
a mystery.

In [ ]:
import numpy as np
import torch
from torch import nn

from ai_power_course.config import scaled, set_seed
from ai_power_course.data import load_energy_data, make_supervised, time_split
from ai_power_course.metrics import mae, rmse
from ai_power_course.models.training import Standardizer, make_loader

set_seed()
torch.set_num_threads(min(4, torch.get_num_threads()))

frame = load_energy_data().frame
HORIZON = 24
train_raw, valid_raw, test_raw = time_split(frame)
X_train, y_train = make_supervised(train_raw, horizon=HORIZON, exogenous=("temperature_c",))
X_valid, y_valid = make_supervised(valid_raw, horizon=HORIZON, exogenous=("temperature_c",))
X_test, y_test = make_supervised(test_raw, horizon=HORIZON, exogenous=("temperature_c",))

# Networks need standardised inputs AND targets, fitted on training data only.
feature_scaler = Standardizer().fit(X_train.to_numpy())
target_scaler = Standardizer().fit(y_train.to_numpy().reshape(-1, 1))
Xs_train = feature_scaler.transform(X_train.to_numpy())
Xs_valid = feature_scaler.transform(X_valid.to_numpy())
Xs_test = feature_scaler.transform(X_test.to_numpy())
ys_train = target_scaler.transform(y_train.to_numpy().reshape(-1, 1))
ys_valid = target_scaler.transform(y_valid.to_numpy().reshape(-1, 1))

N_FEATURES = Xs_train.shape[1]
print(f"{N_FEATURES} standardised features, {len(Xs_train):,} training rows")

### Coding Task 2.1 ★ — A neuron, a loss and one gradient step, in NumPy

A neuron is a weighted sum followed by a non-linearity:

$$y = \sigma(\mathbf{w}^\top \mathbf{x} + b)$$

Training it is: measure the error, work out which direction reduces
it, take a small step that way.

**Your task.** Implement the forward pass of a single neuron, the mean squared
error, and one step of gradient descent on the weights — all in
NumPy, with the derivative written out by hand.

**Requirements**

- `neuron(x, w, b)` applies a sigmoid and accepts `x` of shape `(n_samples, n_features)`, returning shape `(n_samples,)`.
- `mse(y_true, y_pred)` returns a scalar.
- `gradient_step` returns the updated `(w, b)`.
- Use NumPy only. No PyTorch in this task.

<details><summary><b>Hints</b></summary>

- For a linear output (no sigmoid) the MSE gradient is `dL/dw = (2/n) * X.T @ (y_pred - y_true)`.
- Build the sigmoid with `1 / (1 + np.exp(-np.clip(z, -500, 500)))` so a large negative `z` does not overflow.

</details>

**Expected outcome.** The loss printed after the step should be lower than the loss before
it. If it rises, the sign of the gradient is flipped or the learning
rate is far too large.

In [ ]:
def sigmoid(z):
    # TODO: numerically stable sigmoid
    raise NotImplementedError

def neuron(x, w, b):
    """One neuron: weighted sum, then a sigmoid."""
    # TODO
    raise NotImplementedError

def mse(y_true, y_pred):
    # TODO
    raise NotImplementedError

def gradient_step(x, y_true, w, b, learning_rate=0.1):
    """One step of gradient descent on a LINEAR neuron (no sigmoid)."""
    y_pred = x @ w + b
    n = len(y_true)
    # TODO: derivatives of the MSE with respect to w and b
    grad_w = ____
    grad_b = ____
    return w - learning_rate * grad_w, b - learning_rate * grad_b

rng = np.random.default_rng(0)
x_demo = rng.normal(size=(64, 3))
y_demo = x_demo @ np.array([2.0, -1.0, 0.5]) + 0.3
w, b = np.zeros(3), 0.0

print(f"loss before: {mse(y_demo, x_demo @ w + b):.4f}")
for _ in range(50):
    w, b = gradient_step(x_demo, y_demo, w, b, learning_rate=0.1)
print(f"loss after : {mse(y_demo, x_demo @ w + b):.4f}")
print(f"recovered weights: {w.round(3)} (true: [2.0, -1.0, 0.5])")

In [ ]:
_x = np.array([[1.0, 2.0]])
_w, _b = np.array([0.5, -0.25]), 0.1
assert np.isclose(neuron(_x, _w, _b)[0], 1 / (1 + np.exp(-0.1)), atol=1e-9)
assert np.isclose(mse([1.0, 2.0], [1.0, 3.0]), 0.5)
assert np.allclose(w, [2.0, -1.0, 0.5], atol=0.05), "gradient descent did not converge"
print("Basic checks passed.")

### Coding Task 2.2 ★★ — An MLP in PyTorch

Stacking neurons is only worth anything if there is a non-linearity
between the layers. Without one, a composition of linear maps is a
linear map, and a fifty-layer network has exactly the expressive power
of linear regression.

**Your task.** Implement `LoadMLP` as an `nn.Module` with two hidden layers, ReLU
activations and dropout, then confirm it produces the right output
shape.

**Requirements**

- `LoadMLP(n_features, hidden=(128, 64), dropout=0.1)`.
- `forward` takes `(batch, n_features)` and returns `(batch, 1)`.
- Use `nn.Sequential` inside the module.

<details><summary><b>Hints</b></summary>

- The last layer maps to 1 output and must NOT be followed by an activation — the target is an unbounded real number.
- Build the layer list in a loop so the hidden sizes stay configurable.

</details>

**Expected outcome.** A parameter count in the low tens of thousands, and an output of shape
`(batch, 1)`.

In [ ]:
class LoadMLP(nn.Module):
    """A multi-layer perceptron for day-ahead load forecasting."""

    def __init__(self, n_features, hidden=(128, 64), dropout=0.1):
        super().__init__()
        layers = []
        in_features = n_features
        # TODO: for each hidden width, append Linear -> ReLU -> Dropout
        for width in hidden:
            layers += [____, ____, ____]
            in_features = ____
        # TODO: the output layer (no activation)
        layers.append(____)
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # TODO
        pass

set_seed()
model = LoadMLP(N_FEATURES)
print(model)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
_out = model(torch.randn(7, N_FEATURES))
assert _out.shape == (7, 1), f"expected (7, 1), got {tuple(_out.shape)}"
assert any(isinstance(m, nn.ReLU) for m in model.modules()), \
    "without a non-linearity this is just linear regression"
print("Basic checks passed.")

### Coding Task 2.3 ★★ — Write the training loop

Four lines, in a specific order, repeated: zero the gradients,
forward, backward, step. Getting the order wrong produces a model
that trains but learns the wrong thing.

*Reuses what you wrote in 2.2 — do those first.*

**Your task.** Complete the training loop and train the MLP for a few epochs,
tracking training and validation loss.

**Requirements**

- Use `nn.MSELoss()` and `torch.optim.AdamW`.
- Switch between `model.train()` and `model.eval()` correctly.
- Evaluate under `torch.no_grad()`.
- Return the two loss curves.

<details><summary><b>Hints</b></summary>

- `optimizer.zero_grad(set_to_none=True)` must come *before* `loss.backward()`, or you accumulate gradients across batches.
- `make_loader(X, y, batch_size=..., shuffle=True)` is already imported.

</details>

**Expected outcome.** Both curves should fall. If the validation curve turns upward while
the training curve keeps falling, that gap is overfitting — and it is
what early stopping exists to catch.

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total, count = 0.0, 0
    for features, target in loader:
        prediction = model(features)
        loss = loss_fn(prediction, target)
        # TODO: the four lines, in the right order
        ____
        ____
        ____
        total += float(loss.detach()) * len(features)
        count += len(features)
    return total / count

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    # TODO: eval mode, then average the loss over the loader
    raise NotImplementedError

set_seed()
model = LoadMLP(N_FEATURES)
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
train_loader = make_loader(Xs_train, ys_train, batch_size=64, shuffle=True)
valid_loader = make_loader(Xs_valid, ys_valid, batch_size=256)

EPOCHS = scaled(full=12, fast=2)
history = {"train": [], "validation": []}
for epoch in range(EPOCHS):
    history["train"].append(train_epoch(model, train_loader, loss_fn, optimizer))
    history["validation"].append(evaluate(model, valid_loader, loss_fn))
    print(f"epoch {epoch + 1:2d}  train {history['train'][-1]:.5f}  "
          f"validation {history['validation'][-1]:.5f}")

In [ ]:
assert history["train"][-1] < history["train"][0], "the training loss did not fall"
assert len(history["validation"]) == EPOCHS
print("Basic checks passed.")

### Analysis Task 2.4 ★★ — Does the network beat gradient boosting?

Tutorial 02 found the two within a few percent of each other — a gap
smaller than the fold-to-fold spread of cross-validation on this
dataset.

*Reuses what you wrote in 2.3 — do those first.*

**Your task.** Evaluate the trained MLP on the test set in megawatts, compare it
against gradient boosting trained on the same features, and report
both the accuracy gap and the training-time gap.

**Requirements**

- Invert the target standardisation before computing MAE — a score in standardised units is not interpretable.
- Time both fits.
- Print the relative difference in MAE with an explicit direction.

<details><summary><b>Hints</b></summary>

- `target_scaler.inverse_transform(...)` takes the model output back to megawatts.
- Put the model in `eval()` mode before predicting.

</details>

**Expected outcome.** A few percent between them either way, and a much larger factor in
training time. The point of the comparison is the *ratio of effort to
payoff*, not the winner.

In [ ]:
import time

from sklearn.ensemble import HistGradientBoostingRegressor

# TODO 1: MLP predictions in MW
model.eval()
with torch.no_grad():
    mlp_scaled = model(torch.tensor(Xs_test, dtype=torch.float32)).numpy()
mlp_prediction = ____

# TODO 2: gradient boosting on the same features, timed
started = time.perf_counter()
boosting = HistGradientBoostingRegressor(max_iter=200, random_state=0)
boosting.fit(____, ____)
boosting_seconds = time.perf_counter() - started
boosting_prediction = ____

print(f"MLP               MAE {mae(y_test, mlp_prediction):8,.1f} MW")
print(f"gradient boosting MAE {mae(y_test, boosting_prediction):8,.1f} MW"
      f"   ({boosting_seconds:.1f}s to fit)")

gap = mae(y_test, mlp_prediction) / mae(y_test, boosting_prediction) - 1
print(f"\nThe MLP's MAE is {abs(gap):.1%} "
      f"{'higher' if gap > 0 else 'lower'} than gradient boosting's.")

In [ ]:
assert mlp_prediction.shape == y_test.to_numpy().shape, \
    "predictions and targets have different shapes"
assert mae(y_test, mlp_prediction) < 5000, \
    "MAE looks like standardised units — did you invert the target scaler?"
print("Basic checks passed.")

---

## Exercise 03 — RNNs and LSTMs

*Builds on [`03_rnns_and_lstms.ipynb`](03_rnns_and_lstms.ipynb).*

Until now the models saw a *table* whose columns happened to be called
`lag24`. Here they see the sequence itself. That change is what lets a model
accept a longer history, a different site, or a different variable without
being rebuilt.

In [ ]:
import numpy as np
import torch
from torch import nn

from ai_power_course.config import scaled, set_seed
from ai_power_course.data import load_energy_data, time_split
from ai_power_course.metrics import mae, rmse
from ai_power_course.models.training import Standardizer, TrainConfig, make_loader, train_model

set_seed()
torch.set_num_threads(min(4, torch.get_num_threads()))

frame = load_energy_data().frame
train_raw, valid_raw, test_raw = time_split(frame)

CONTEXT, HORIZON = 168, 24
# Stride the training windows: consecutive windows overlap by 167 of 168
# hours, so every fourth origin keeps full coverage at a quarter of the cost.
TRAIN_STRIDE = scaled(full=4, fast=24)
print(f"context {CONTEXT} h, horizon {HORIZON} h, training stride {TRAIN_STRIDE}")

### Coding Task 3.1 ★★ — Slice a series into supervised windows

A sequence model consumes fixed-length windows: `context` steps of
history in, `horizon` steps of future out. Building them is pure index
arithmetic, and getting it wrong by one step is the most common bug in
the whole field.

**Your task.** Implement `create_sequences`, returning inputs of shape
`(n_windows, context, 1)` and targets of shape `(n_windows, horizon)`.

**Requirements**

- Window `i` covers `series[i : i + context]`.
- Its target is `series[i + context : i + context + horizon]`.
- Support a `stride` so windows can be subsampled.
- Return float32 arrays.

<details><summary><b>Hints</b></summary>

- The last valid start index is `len(series) - context - horizon`.
- `np.stack([...])` over a list comprehension is clear and fast enough here.

</details>

**Expected outcome.** The assertion cell should pass, and the window count should be roughly
`(len(series) - context - horizon) / stride`.

In [ ]:
def create_sequences(series, context, horizon, stride=1):
    """Slice a 1-D series into (context, horizon) supervised windows.

    Returns
    -------
    x : (n_windows, context, 1) float32
    y : (n_windows, horizon)    float32
    """
    values = np.asarray(series, dtype=np.float32).reshape(-1)
    # TODO 1: the last index at which a full window plus its target fits
    last_start = ____
    if last_start < 0:
        raise ValueError("series is too short for this context and horizon")
    starts = np.arange(0, last_start + 1, stride)
    # TODO 2: build x and y
    x = ____
    y = ____
    return x, y

x_train, y_train_seq = create_sequences(
    train_raw.load_mw, CONTEXT, HORIZON, stride=TRAIN_STRIDE)
x_valid, y_valid_seq = create_sequences(
    valid_raw.load_mw, CONTEXT, HORIZON, stride=TRAIN_STRIDE)
x_test, y_test_seq = create_sequences(test_raw.load_mw, CONTEXT, HORIZON, stride=1)

print(f"x_train {x_train.shape}  y_train {y_train_seq.shape}")
print(f"x_test  {x_test.shape}  y_test  {y_test_seq.shape}")

In [ ]:
_v = np.arange(100, dtype=np.float32)
_x, _y = create_sequences(_v, context=10, horizon=3)
assert _x.shape == (88, 10, 1), f"got {_x.shape}, expected (88, 10, 1)"
assert _y.shape == (88, 3), f"got {_y.shape}, expected (88, 3)"
assert np.allclose(_x[5, :, 0], np.arange(5, 15)), "window contents are misaligned"
assert np.allclose(_y[5], np.arange(15, 18)), "targets are misaligned"
print("Basic checks passed.")

### Analysis Task 3.2 ★ — Tensor shapes through a recurrent model

Most time lost to sequence models is lost to shapes. Predicting them
before running anything is a skill worth ten debugging sessions.

**Your task.** Fill in the expected shape at each stage of a forward pass, then check
your answers against a real `nn.LSTM`.

**Requirements**

- Write each expected shape as a tuple before running the check.
- Explain in one line why the hidden state has no sequence axis.

<details><summary><b>Hints</b></summary>

- `nn.LSTM(..., batch_first=True)` returns `(output, (h_n, c_n))`.
- `h_n` has shape `(num_layers, batch, hidden_size)` regardless of `batch_first`.

</details>

**Expected outcome.** All four assertions pass. The output keeps the sequence axis; the final
hidden state does not, because it is a summary of the whole sequence.

In [ ]:
BATCH, HIDDEN = 8, 32

# TODO: predict each shape as a tuple, then run the cell.
expected_input_shape = ____          # what goes into the LSTM
expected_output_shape = ____         # the per-step outputs
expected_hidden_shape = ____         # the final hidden state h_n
expected_prediction_shape = ____     # after a Linear(HIDDEN, HORIZON) head

lstm = nn.LSTM(input_size=1, hidden_size=HIDDEN, num_layers=1, batch_first=True)
head = nn.Linear(HIDDEN, HORIZON)
sample = torch.randn(BATCH, CONTEXT, 1)

output, (h_n, c_n) = lstm(sample)
prediction = head(h_n[-1])

assert tuple(sample.shape) == expected_input_shape
assert tuple(output.shape) == expected_output_shape
assert tuple(h_n.shape) == expected_hidden_shape
assert tuple(prediction.shape) == expected_prediction_shape
print("All four shapes correct.")

### Coding Task 3.3 ★★ — Build and train an LSTM forecaster

The LSTM's cell state is an *additive* path through time, which is why
a gradient can survive hundreds of steps instead of decaying
geometrically. That is the whole reason it displaced the vanilla RNN.

*Reuses what you wrote in 3.1 — do those first.*

**Your task.** Implement `LoadLSTM` as a sequence-to-one model that decodes the final
hidden state into the whole horizon, then train it on the standardised
windows.

**Requirements**

- `LoadLSTM(n_channels, hidden_size, horizon)`.
- `forward` takes `(batch, context, n_channels)` and returns `(batch, horizon)`.
- Train with the package's `train_model`, which handles early stopping and restores the best weights.

<details><summary><b>Hints</b></summary>

- Standardise using statistics from the *training* windows only.
- Decode `h_n[-1]` with a `nn.Linear(hidden_size, horizon)`.

</details>

**Expected outcome.** A validation loss that falls and then flattens. Early stopping should
fire before the epoch budget runs out.

In [ ]:
scaler = Standardizer().fit(x_train.reshape(-1, 1))
xs_train = scaler.transform(x_train)
xs_valid = scaler.transform(x_valid)
xs_test = scaler.transform(x_test)

target_scaler = Standardizer().fit(y_train_seq)
ys_train = target_scaler.transform(y_train_seq)
ys_valid = target_scaler.transform(y_valid_seq)

class LoadLSTM(nn.Module):
    def __init__(self, n_channels=1, hidden_size=64, horizon=HORIZON):
        super().__init__()
        # TODO: an LSTM and a linear head
        self.lstm = ____
        self.head = ____

    def forward(self, x):
        # TODO: run the LSTM, take the final hidden state, decode it
        ____
        return ____

set_seed()
lstm_model = LoadLSTM()
history = train_model(
    lstm_model,
    make_loader(xs_train, ys_train, batch_size=64, shuffle=True),
    make_loader(xs_valid, ys_valid, batch_size=256),
    loss_fn=nn.MSELoss(),
    config=TrainConfig(epochs=scaled(full=25, fast=2), learning_rate=2e-3,
                       patience=5, verbose=False),
)
print(f"best epoch {history.best_epoch}, "
      f"validation MSE {history.best_val_loss:.5f}, {history.seconds:.0f}s")

In [ ]:
_out = lstm_model(torch.randn(3, CONTEXT, 1))
assert _out.shape == (3, HORIZON), f"got {tuple(_out.shape)}, expected (3, {HORIZON})"
assert history.train_loss[-1] < history.train_loss[0], "training loss did not fall"
print("Basic checks passed.")

### Analysis Task 3.4 ★★★ — Where does the LSTM actually fail?

An aggregate MAE is an average over very different regimes. Tutorial
03 tested two hypotheses about *when* the model struggles, and one of
them was wrong — which is worth more than the one that held.

*Reuses what you wrote in 3.1, 3.3 — do those first.*

**Your task.** Score the LSTM against persistence on the test windows, then test
whether the error is predicted by (a) how volatile the recent past was
and (b) how unusual the target hour is for its calendar slot.

**Requirements**

- Report MAE at `t+24` for both the LSTM and persistence.
- Compute a correlation for each hypothesis.
- State which hypothesis survived, using the numbers.

<details><summary><b>Hints</b></summary>

- Recent volatility: the largest absolute hour-to-hour change in the last 24 hours of the context window.
- Unusualness: the absolute deviation of the target from the training mean for that (hour, weekday) slot.

</details>

**Expected outcome.** One of the two correlations should be clearly stronger. Report the one
that failed as well — a negative result about your own hypothesis is a
result.

In [ ]:
import pandas as pd

lstm_model.eval()
with torch.no_grad():
    scaled_prediction = lstm_model(torch.tensor(xs_test, dtype=torch.float32)).numpy()
lstm_prediction = target_scaler.inverse_transform(scaled_prediction)

truth = y_test_seq[:, -1]                  # the load at t+24
lstm_point = lstm_prediction[:, -1]
persistence_point = x_test[:, -1, 0]       # the load at the origin

print(f"LSTM        MAE {mae(truth, lstm_point):8,.1f} MW")
print(f"persistence MAE {mae(truth, persistence_point):8,.1f} MW")

errors = np.abs(lstm_point - truth)

# TODO 1: recent volatility — largest |change| in the last 24 h of context
recent_volatility = ____

# TODO 2: unusualness — |target - training mean for that (hour, weekday)|
target_index = test_raw.index[CONTEXT + HORIZON - 1 :][: len(truth)]
profile = train_raw.load_mw.groupby(
    [train_raw.index.hour, train_raw.index.dayofweek]).mean()
expected_load = np.array([
    profile.get((h, d), float(train_raw.load_mw.mean()))
    for h, d in zip(target_index.hour, target_index.dayofweek, strict=True)
])
unusualness = ____

for label, driver in [("recent volatility", recent_volatility),
                      ("unusualness of the target", unusualness)]:
    r = np.corrcoef(driver, errors)[0, 1]
    print(f"{label:28s} r = {r:+.3f}")

In [ ]:
from ai_power_course.config import fast_mode

assert recent_volatility.shape == errors.shape
assert unusualness.shape == errors.shape
assert np.isfinite(errors).all(), "some predictions are not finite"
if fast_mode():
    print("Reduced configuration: skipping the accuracy check — two epochs")
    print("is not enough training to beat persistence.")
else:
    assert mae(truth, lstm_point) < mae(truth, persistence_point), \
        "the LSTM should beat persistence at a 24 h horizon"
print("Basic checks passed.")

---

## Exercise 04 — Representation Learning

*Builds on [`04_representation_learning.ipynb`](04_representation_learning.ipynb).*

The hinge of the course. Everything so far started from a task. Here you
train a model with **no labels and no task**, then discover its internal
representation is useful for a question it was never shown.

That is the mechanism behind every foundation model in chapters 09 and 10.

In [ ]:
import numpy as np
import torch
from torch import nn

from ai_power_course.config import fast_mode, scaled, set_seed
from ai_power_course.data import load_energy_data, time_split
from ai_power_course.metrics import mae, r2

set_seed()
torch.set_num_threads(min(4, torch.get_num_threads()))

frame = load_energy_data().frame
train_raw, valid_raw, test_raw = time_split(frame)

def daily_profiles(part, column="residual_load_mw"):
    """Reshape a split into per-day profiles, normalised to shape only."""
    series = part[column]
    complete = [d for d, g in series.groupby(series.index.date) if len(g) == 24]
    matrix = np.stack([series[series.index.date == d].to_numpy() for d in complete])
    level = matrix.mean(axis=1, keepdims=True)
    spread = matrix.std(axis=1, keepdims=True) + 1e-8
    normalised = ((matrix - level) / spread).astype(np.float32)
    import pandas as pd
    return normalised[:, :, None], pd.to_datetime(complete)

profiles_train, dates_train = daily_profiles(train_raw)
profiles_valid, dates_valid = daily_profiles(valid_raw)
profiles_test, dates_test = daily_profiles(test_raw)
print(f"pretraining profiles {profiles_train.shape} — and not one label among them")

### Coding Task 4.1 ★★ — Mask part of a profile

Self-supervision manufactures a supervised problem out of the data's
own structure: hide part of the input and predict it from the rest.

*Which* part you hide is a modelling decision. Scattered single hours
can be interpolated from their neighbours; a contiguous block cannot,
and a contiguous block is what a communication outage actually looks
like.

**Your task.** Implement both masking strategies and a masked reconstruction loss
that scores **only the hidden positions**.

**Requirements**

- `random_mask(shape, ratio)` returns a boolean tensor, `True` = hidden.
- `block_mask(shape, block)` hides exactly one contiguous run per sample.
- `masked_loss(prediction, target, mask)` ignores visible positions.
- All three accept a `torch.Generator` for reproducibility.

<details><summary><b>Hints</b></summary>

- For the block mask, draw one start per sample and compare against `torch.arange(length)` broadcast over the batch.
- Weight the squared error by the mask and divide by the number of masked *elements*, not by the batch size.

</details>

**Expected outcome.** The block mask should hide exactly `block` positions per sample, and
they should be consecutive. The loss on a perfect reconstruction
should be zero even when most positions are wrong outside the mask.

In [ ]:
def random_mask(shape, ratio=0.3, generator=None):
    """Independent Bernoulli mask. True means hidden."""
    # TODO
    raise NotImplementedError

def block_mask(shape, block=6, generator=None):
    """Hide exactly one contiguous run of `block` positions per sample."""
    batch, length = shape
    # TODO 1: one random start per sample, in [0, length - block]
    starts = ____
    # TODO 2: compare a position row against the starts
    positions = torch.arange(length).unsqueeze(0)
    return ____

def masked_loss(prediction, target, mask, eps=1e-8):
    """MSE over the masked positions only."""
    # TODO: weight by the mask, then normalise by the number of masked values
    raise NotImplementedError

gen = torch.Generator().manual_seed(0)
m_random = random_mask((100, 24), ratio=0.3, generator=gen)
m_block = block_mask((100, 24), block=6, generator=gen)
print(f"random mask hides {m_random.float().mean():.1%} of positions")
print(f"block mask hides  {m_block.sum(dim=1).float().mean():.1f} positions per day")

In [ ]:
_g = torch.Generator().manual_seed(1)
_b = block_mask((50, 24), block=6, generator=_g)
assert (_b.sum(dim=1) == 6).all(), "each sample must hide exactly `block` positions"
for _row in _b:
    assert (torch.where(_row)[0].diff() == 1).all(), "the block must be contiguous"
_p, _t = torch.zeros(2, 4, 1), torch.ones(2, 4, 1)
_m = torch.tensor([[True, False, False, False], [False, False, False, True]])
assert abs(float(masked_loss(_p, _t, _m)) - 1.0) < 1e-6, \
    "the loss must ignore visible positions"
print("Basic checks passed.")

### Coding Task 4.2 ★★ — A masked autoencoder for daily profiles

A plain autoencoder can cheat: with a wide enough bottleneck it
approximates the identity. Masking removes that escape route — you
cannot copy what you cannot see.

*Reuses what you wrote in 4.1 — do those first.*

**Your task.** Build an encoder that compresses a 24-hour profile to a small latent
vector and a decoder that expands it back, then pretrain them on the
masking objective from task 4.1.

**Requirements**

- `ProfileAutoencoder(length=24, latent_dim=8)`.
- `forward(x, mask)` replaces the masked positions with a *learned* mask token before encoding.
- `embed(x)` returns the latent code with no masking, in eval mode.
- Pretraining must use no labels of any kind.

<details><summary><b>Hints</b></summary>

- `nn.Parameter(torch.zeros(1))` gives you a learnable scalar to substitute at masked positions — BERT's `[MASK]`, one number wide.
- `torch.where(mask.unsqueeze(-1), token.expand_as(x), x)` does the substitution without a loop.

</details>

**Expected outcome.** The reconstruction loss should fall by an order of magnitude. The
reconstructions plotted afterwards should follow the shape of the
hidden stretch, not just interpolate a straight line across it.

In [ ]:
LATENT = 8

class ProfileAutoencoder(nn.Module):
    def __init__(self, length=24, latent_dim=LATENT, hidden=64):
        super().__init__()
        self.length = length
        # TODO 1: encoder — flatten, then down to latent_dim
        self.encoder = nn.Sequential(
            nn.Flatten(),
            ____,
            nn.GELU(),
            ____,
        )
        # TODO 2: decoder — latent_dim back up to `length`
        self.decoder = nn.Sequential(
            ____,
            nn.GELU(),
            ____,
            nn.Unflatten(1, (length, 1)),
        )
        # TODO 3: the learned mask token
        self.mask_token = ____

    def forward(self, x, mask=None):
        if mask is not None:
            x = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(x), x)
        return self.decoder(self.encoder(x))

    @torch.no_grad()
    def embed(self, x):
        self.eval()
        return self.encoder(x)

set_seed()
autoencoder = ProfileAutoencoder()
optimizer = torch.optim.AdamW(autoencoder.parameters(), lr=2e-3)
gen = torch.Generator().manual_seed(0)
inputs = torch.tensor(profiles_train)

pretrain_curve = []
for epoch in range(scaled(full=60, fast=5)):
    autoencoder.train()
    order = torch.randperm(len(inputs), generator=gen)
    epoch_loss, seen = 0.0, 0
    for start in range(0, len(order), 64):
        batch = inputs[order[start : start + 64]]
        half = len(batch) // 2
        mask = torch.cat([
            random_mask((half, 24), ratio=0.3, generator=gen),
            block_mask((len(batch) - half, 24), block=6, generator=gen),
        ])
        # TODO 4: forward, loss, backward, step
        ____
        ____
        ____
        ____
        ____
        epoch_loss += float(loss.detach()) * len(batch)
        seen += len(batch)
    pretrain_curve.append(epoch_loss / seen)
print(f"masked reconstruction loss: "
      f"{pretrain_curve[0]:.4f} -> {pretrain_curve[-1]:.4f}")

In [ ]:
assert pretrain_curve[-1] < pretrain_curve[0], "the reconstruction loss did not fall"
_z = autoencoder.embed(torch.tensor(profiles_valid[:5]))
assert _z.shape == (5, LATENT), f"expected (5, {LATENT}), got {tuple(_z.shape)}"
print("Basic checks passed.")

### Analysis Task 4.3 ★★★ — Probe what the encoder learned without labels

The strict test of a representation is not whether a 2-D projection
*looks* clustered — it is whether a **linear** model can read a factor
out of it. Those two tests disagree more often than people expect.

*Reuses what you wrote in 4.2 — do those first.*

**Your task.** Project the embeddings to two dimensions with PCA and colour them by a
factor the encoder never saw, then run a linear probe for three
different factors and report which are recoverable.

**Requirements**

- Colour the PCA scatter by the daily PV share.
- Probe at least three factors with a ridge regression, reporting R².
- Use a random split for the probe, and say in one line why that is appropriate here when it was not in chapter 01.

<details><summary><b>Hints</b></summary>

- PV share for a day: that day's PV energy divided by its load energy.
- A negative R² means the probe is worse than predicting the mean — that factor is not linearly present.

</details>

**Expected outcome.** Sun-related factors should come out clearly. At least one factor should
*fail*, and the interesting part of the task is explaining why.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

embeddings = autoencoder.embed(torch.tensor(profiles_train)).numpy()
print(f"embeddings {embeddings.shape}")

daily = frame.groupby(frame.index.date)[["pv_mw", "load_mw"]].sum()
pv_share = np.array([
    daily.loc[d.date(), "pv_mw"] / max(daily.loc[d.date(), "load_mw"], 1e-9)
    for d in dates_train
])
daily_temperature = frame.groupby(frame.index.date).temperature_c.mean()
temperature = np.array([daily_temperature[d.date()] for d in dates_train])
is_weekend = (dates_train.dayofweek >= 5).astype(float)

# TODO 1: PCA to two components and a scatter coloured by pv_share
coords = ____
fig, ax = plt.subplots(figsize=(5.4, 4.2))
scatter = ax.scatter(____, ____, c=____, s=9, cmap="viridis")
plt.colorbar(scatter, ax=ax, label="daily PV share")
ax.set_xlabel(____)
ax.set_ylabel(____)
ax.set_title(____)
plt.show()

# TODO 2: a linear probe for each factor
def linear_probe(features, target, name, seed=0):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(features))
    cut = int(0.7 * len(order))
    model = Ridge(alpha=1.0).fit(____, ____)
    score = r2(____, ____)
    print(f"  {name:28s} R2 = {score:6.3f}")
    return score

print("Linear readout from the 8-dimensional embedding:")
for name, values in [("daily PV share", pv_share),
                     ("mean temperature", temperature),
                     ("is it a weekend?", is_weekend)]:
    linear_probe(embeddings, values, name)

In [ ]:
assert coords.shape == (len(embeddings), 2)
assert scores["daily PV share"] > 0.3, \
    "PV share should be clearly readable — check the pretraining loss fell"
print("Basic checks passed.")

### Coding Task 4.4 ★★★ — Reuse the frozen encoder with very few labels

Pretraining does not buy accuracy. It buys **label efficiency** — and
whether that is worth anything depends entirely on whether labels are
your scarce resource. In power systems they usually are: measurements
are everywhere, annotations are not.

*Reuses what you wrote in 4.2 — do those first.*

**Your task.** Freeze the encoder, fit a ridge regression on its embeddings to
predict the daily PV share, and compare against the same ridge fitted
on the raw 24 values — across several label budgets.

**Requirements**

- The encoder must stay frozen; only the head is fitted.
- Evaluate on the test year, never on the labels used for fitting.
- Sweep at least three label budgets.
- Average over a few random draws so the small-budget numbers are not one lucky sample.

<details><summary><b>Hints</b></summary>

- `autoencoder.embed` is already decorated with `torch.no_grad`.
- Use the validation year as the labelled pool and the test year for evaluation, so neither was seen during pretraining.

</details>

**Expected outcome.** The frozen encoder should help most at the smallest budget. Expect the
advantage to be modest — 24 numbers is a small input, and the raw
baseline is strong. Report the size of the effect honestly.

In [ ]:
import pandas as pd
from sklearn.linear_model import Ridge

def pv_share_for(dates):
    daily = frame.groupby(frame.index.date)[["pv_mw", "load_mw"]].sum()
    return np.array([
        daily.loc[d.date(), "pv_mw"] / max(daily.loc[d.date(), "load_mw"], 1e-9)
        for d in dates
    ])

y_pool = pv_share_for(dates_valid)      # labelled pool
y_eval = pv_share_for(dates_test)       # evaluation

# TODO 1: embeddings for the pool and the evaluation set
z_pool = ____
z_eval = ____
raw_pool = profiles_valid[:, :, 0]
raw_eval = profiles_test[:, :, 0]

def score_at(n_labels, features_pool, features_eval, seed):
    rng = np.random.default_rng(seed)
    chosen = rng.choice(len(y_pool), size=n_labels, replace=False)
    # TODO 2: fit ridge on the chosen labels, score on the evaluation set
    model = ____
    return mae(____, ____)

budgets = [10, 30] if fast_mode() else [10, 30, 100]
rows = []
for n in budgets:
    rows.append({
        "labels": n,
        "frozen encoder": np.mean(
            [score_at(n, z_pool, z_eval, s) for s in range(3)]),
        "raw 24 values": np.mean(
            [score_at(n, raw_pool, raw_eval, s) for s in range(3)]),
    })
efficiency = pd.DataFrame(rows).set_index("labels")
display(efficiency.round(4))

In [ ]:
assert z_pool.shape[1] == LATENT, "the probe should use the latent code"
assert (efficiency > 0).all().all(), "MAE must be positive"
print("Basic checks passed.")

---

## Exercise 05 — Attention

*Builds on [`05_attention.ipynb`](05_attention.ipynb).*

The central implementation exercise of the course. You will write scaled
dot-product attention from the equation, in NumPy, and only compare it
against PyTorch's version once yours works.

Do not use `nn.MultiheadAttention` or
`torch.nn.functional.scaled_dot_product_attention` for task 5.1. The whole
point is that you can write it.

In [ ]:
import math

import numpy as np
import torch

from ai_power_course.config import set_seed
from ai_power_course.data import load_energy_data, time_split

set_seed()
frame = load_energy_data().frame
train_raw, valid_raw, test_raw = time_split(frame)
print("Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V")

### Coding Task 5.1 ★★ — Scaled dot-product attention from the equation

Attention is a soft, differentiable dictionary lookup. A **query**
asks what it is looking for, each **key** advertises what its position
offers, and each **value** is what that position returns if chosen.
Instead of retrieving one value, return a weighted average of all of
them:

$$S = \frac{QK^\top}{\sqrt{d_k}}, \qquad
  A = \mathrm{softmax}(S), \qquad
  Y = AV$$

**Your task.** Implement the three steps in NumPy: score, normalise, aggregate.
Implement the softmax yourself too, numerically stably.

**Requirements**

- `softmax(x, axis=-1)` must subtract the max before exponentiating.
- `scaled_dot_product_attention(Q, K, V, mask=None)` returns `(output, weights)`.
- `Q` is `(n_queries, d_k)`, `K` is `(n_keys, d_k)`, `V` is `(n_keys, d_v)`.
- `mask` is boolean with `True` = *may attend*; masked scores become `-inf` **before** the softmax.
- NumPy only. No PyTorch attention helpers.

<details><summary><b>Hints</b></summary>

- Each row of the weight matrix is a probability distribution over the keys, so it must sum to 1 along the last axis.
- Masking after the softmax would leave the surviving weights not summing to one — apply it to the scores.

</details>

**Expected outcome.** Output of shape `(n_queries, d_v)`, weights of shape
`(n_queries, n_keys)` with rows summing to 1, and agreement with
PyTorch to floating-point precision.

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    # TODO: subtract the max along `axis` before exponentiating
    raise NotImplementedError

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V.

    Returns
    -------
    output  : (n_queries, d_v)
    weights : (n_queries, n_keys)
    """
    d_k = Q.shape[-1]
    # TODO 1: the scaled score matrix
    scores = ____
    # TODO 2: apply the mask, if given, BEFORE the softmax
    if mask is not None:
        scores = ____
    # TODO 3: normalise, then aggregate the values
    weights = ____
    output = ____
    return output, weights

rng = np.random.default_rng(0)
Q = rng.normal(size=(6, 8))
K = rng.normal(size=(6, 8))
V = rng.normal(size=(6, 4))
output, weights = scaled_dot_product_attention(Q, K, V)
print(f"output {output.shape}   weights {weights.shape}")
print(f"every row of the weight matrix sums to 1: "
      f"{np.allclose(weights.sum(axis=-1), 1.0)}")

In [ ]:
def check_attention_function(fn):
    rng = np.random.default_rng(7)
    q, k, v = (rng.normal(size=(5, 8)) for _ in range(3))
    out, w = fn(q, k, v)
    assert out.shape == (5, 8), f"output should be (5, 8), got {out.shape}"
    assert w.shape == (5, 5), f"weights should be (5, 5), got {w.shape}"
    assert np.allclose(w.sum(axis=-1), 1.0, atol=1e-6), "rows must sum to 1"
    assert (w >= 0).all(), "attention weights cannot be negative"
    reference = torch.nn.functional.scaled_dot_product_attention(
        *(torch.tensor(a).unsqueeze(0) for a in (q, k, v))
    ).squeeze(0).numpy()
    assert np.allclose(out, reference, atol=1e-10), "disagrees with PyTorch"
    print("Basic checks passed — and it matches PyTorch to 1e-10.")

check_attention_function(scaled_dot_product_attention)

### Analysis Task 5.2 ★★ — Why divide by the square root of d_k?

For random `q` and `k` with unit variance, the dot product
$\sum_{i=1}^{d_k} q_i k_i$ has variance $d_k$. At $d_k = 256$ the
scores have a standard deviation of 16, and a softmax over scores that
spread is effectively an `argmax`.

*Reuses what you wrote in 5.1 — do those first.*

**Your task.** Measure the effect: for several key dimensions, compare the standard
deviation of the scores and the largest softmax weight, with and
without the scaling.

**Requirements**

- Sweep at least four values of `d_k` spanning two orders of magnitude.
- Report the score spread and the maximum attention weight for both variants.
- Say in one line why a saturated softmax stops the model learning.

<details><summary><b>Hints</b></summary>

- The scaled scores should have a standard deviation close to 1 regardless of `d_k` — that is the whole point.
- A maximum weight near 1.0 means one key has taken everything.

</details>

**Expected outcome.** Unscaled, the maximum weight should march towards 1.0 as `d_k` grows.
Scaled, it should stay small and roughly constant.

In [ ]:
import pandas as pd

rows = []
for d_k in [4, 16, 64, 256, 1024]:
    rng = np.random.default_rng(1)
    q = rng.normal(size=(1, d_k))
    k = rng.normal(size=(64, d_k))
    # TODO: scores with and without the 1/sqrt(d_k) factor
    unscaled = ____
    scaled_scores = ____
    rows.append({
        "d_k": d_k,
        "score std (unscaled)": unscaled.std(),
        "score std (scaled)": scaled_scores.std(),
        "max weight (unscaled)": softmax(unscaled).max(),
        "max weight (scaled)": softmax(scaled_scores).max(),
    })
display(pd.DataFrame(rows).set_index("d_k").round(3))

In [ ]:
assert table.loc[1024, "max weight (unscaled)"] > 0.9, \
    "without scaling, a large d_k should saturate the softmax"
assert table.loc[1024, "max weight (scaled)"] < 0.3, \
    "with scaling, the distribution should stay usable"
print("Basic checks passed.")

### Coding Task 5.3 ★★ — Causal masking and an attention heatmap

One line separates a model that fills in blanks from a model that
predicts the future: a lower-triangular mask, so position $i$ can
attend only to $j \le i$.

That single change is the whole difference between BERT and GPT.

*Reuses what you wrote in 5.1 — do those first.*

**Your task.** Build a causal mask, apply it through your attention function, and
plot both the unmasked and masked weight matrices side by side.

**Requirements**

- `causal_mask(n)` returns an `(n, n)` boolean array, `True` = allowed.
- Both heatmaps need a colourbar, axis labels and a title.
- Assert that the masked weights above the diagonal are exactly zero.

<details><summary><b>Hints</b></summary>

- `np.tril(np.ones((n, n), dtype=bool))` is the whole mask.
- `np.triu_indices(n, k=1)` selects the strictly upper triangle for the assertion.

</details>

**Expected outcome.** The masked heatmap should be strictly lower-triangular with exact
zeros above the diagonal, and its rows should still sum to one.

In [ ]:
import matplotlib.pyplot as plt

def causal_mask(n):
    """Lower-triangular boolean mask: True where attention is allowed."""
    # TODO
    raise NotImplementedError

SEQ = 12
rng = np.random.default_rng(3)
Q = rng.normal(size=(SEQ, 16))
K = rng.normal(size=(SEQ, 16))
V = rng.normal(size=(SEQ, 16))

_, weights_open = scaled_dot_product_attention(Q, K, V)
# TODO: the same, with the causal mask
_, weights_causal = ____

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, matrix, title in [
    (axes[0], weights_open, "Bidirectional: every position sees every other"),
    (axes[1], weights_causal, "Causal: position i sees only j <= i"),
]:
    image = ax.imshow(matrix, cmap="magma", aspect="auto")
    plt.colorbar(image, ax=ax, label="attention weight")
    ax.set_xlabel("key position (attended to)")
    ax.set_ylabel("query position")
    ax.set_title(title, fontsize=10)
fig.tight_layout()
plt.show()

assert weights_causal[np.triu_indices(SEQ, k=1)].max() == 0.0
print("No attention leaks into the future.")

In [ ]:
_m = causal_mask(5)
assert _m.dtype == bool and _m.shape == (5, 5)
assert _m[0].sum() == 1 and _m[4].sum() == 5, "row i must allow exactly i+1 keys"
assert not _m[0, 1], "position 0 must not see position 1"
print("Basic checks passed.")

### Reflection Question 5.4 ★★★ — Is a large attention weight an explanation?

Attention heatmaps are extremely persuasive and routinely
over-interpreted. Tutorial 05 replaced a trained model's attention
with a uniform average and measured what actually changed.

**Your task.** Give three concrete reasons why a large attention weight does not, on
its own, establish that the attended position caused the prediction.
Then describe one experiment that would produce *evidence* rather than
a picture.

**Expected outcome.** A strong answer distinguishes "the model looked here" from "this
changed the answer", and proposes an ablation with a number attached.

**Reason 1.** …

**Reason 2.** …

**Reason 3.** …

**An experiment that would produce evidence.** …

---

## Exercise 06 — Transformers

*Builds on [`06_transformers.ipynb`](06_transformers.ipynb).*

Attention is a layer. A Transformer is an architecture: multi-head attention,
a position-wise feed-forward network, residual connections, layer
normalisation and — because attention has no notion of order — positional
information.

Here the tokens are **hours**, not words. That is the point of this chapter.

In [ ]:
import math

import numpy as np
import torch
from torch import nn

from ai_power_course.config import scaled, set_seed
from ai_power_course.data import load_energy_data, time_split
from ai_power_course.metrics import mae
from ai_power_course.models.training import (
    Standardizer, TrainConfig, make_loader, train_model,
)

set_seed()
torch.set_num_threads(min(4, torch.get_num_threads()))

frame = load_energy_data().frame
train_raw, valid_raw, test_raw = time_split(frame)

CONTEXT, HORIZON = 168, 24
STRIDE = scaled(full=6, fast=36)

def windows(part, context=CONTEXT, horizon=HORIZON, stride=1):
    values = part.load_mw.to_numpy(dtype=np.float32)
    starts = np.arange(0, len(values) - context - horizon + 1, stride)
    x = np.stack([values[s : s + context] for s in starts])[:, :, None]
    y = np.stack([values[s + context : s + context + horizon] for s in starts])
    return x.astype(np.float32), y.astype(np.float32)

x_train, y_train = windows(train_raw, stride=STRIDE)
x_valid, y_valid = windows(valid_raw, stride=STRIDE)
x_test, y_test = windows(test_raw, stride=4)

x_scaler = Standardizer().fit(x_train.reshape(-1, 1))
y_scaler = Standardizer().fit(y_train)
xs_train, xs_valid, xs_test = (x_scaler.transform(a) for a in (x_train, x_valid, x_test))
ys_train, ys_valid = (y_scaler.transform(a) for a in (y_train, y_valid))
print(f"{len(xs_train):,} training windows of {CONTEXT} hours -> {HORIZON} hours")

### Coding Task 6.1 ★★ — Multi-head attention

One attention head computes one weighted average, so it can express
one kind of relationship. Splitting `d_model` into `h` heads of width
`d_model / h` lets the layer attend to several patterns at once — the
same hour yesterday, the same hour last week, the recent trend — at no
extra cost, because the head width shrinks as the count grows.

**Your task.** Implement multi-head self-attention in PyTorch: project to Q, K and V,
split into heads, attend, concatenate, project out.

**Requirements**

- `MultiHeadSelfAttention(d_model, n_heads)`, with `d_model` divisible by `n_heads`.
- `forward(x)` takes `(batch, length, d_model)` and returns `(output, weights)` with weights `(batch, n_heads, length, length)`.
- Reuse the maths from task 5.1. Do **not** call `nn.MultiheadAttention` or `F.scaled_dot_product_attention`.
- Assert `d_model % n_heads == 0` in `__init__`.

<details><summary><b>Hints</b></summary>

- One `nn.Linear(d_model, 3 * d_model)` produces Q, K and V together; `.chunk(3, dim=-1)` splits them.
- Splitting heads is a reshape plus a transpose: `(B, L, H, d_head).transpose(1, 2)` -> `(B, H, L, d_head)`.
- Merging them back is the same two operations reversed, and needs `.contiguous()` before the final `.view`.

</details>

**Expected outcome.** Output shape equal to the input shape, weights summing to one along the
last axis, and — for a single head — exact agreement with PyTorch's
fused kernel.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        # TODO 1: guard the divisibility, store d_head
        assert ____, "d_model must be divisible by n_heads"
        self.d_model, self.n_heads = d_model, n_heads
        self.d_head = ____
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, t):
        batch, length, _ = t.shape
        # TODO 2: (B, L, d_model) -> (B, H, L, d_head)
        return ____

    def _merge_heads(self, t):
        batch, heads, length, d_head = t.shape
        # TODO 3: (B, H, L, d_head) -> (B, L, d_model)
        return ____

    def forward(self, x, mask=None):
        q, k, v = (self._split_heads(p) for p in self.qkv(x).chunk(3, dim=-1))
        # TODO 4: scores, mask, softmax, aggregate
        scores = ____
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = ____
        attended = ____
        return self.out_proj(self._merge_heads(attended)), weights

set_seed()
layer = MultiHeadSelfAttention(d_model=64, n_heads=4)
sample = torch.randn(2, 12, 64)
output, weights = layer(sample)
print(f"input {tuple(sample.shape)} -> output {tuple(output.shape)}")
print(f"weights {tuple(weights.shape)} (batch, heads, query, key)")

In [ ]:
assert output.shape == sample.shape, "attention must preserve the shape"
assert weights.shape == (2, 4, 12, 12)
assert torch.allclose(weights.sum(dim=-1), torch.ones(2, 4, 12), atol=1e-5)

# A single head must reproduce PyTorch's fused kernel exactly.
_single = MultiHeadSelfAttention(d_model=16, n_heads=1)
_x = torch.randn(1, 5, 16)
_q, _k, _v = _single.qkv(_x).chunk(3, dim=-1)
_reference = torch.nn.functional.scaled_dot_product_attention(_q, _k, _v)
_ours, _ = _single(_x)
assert torch.allclose(_single.out_proj(_reference), _ours, atol=1e-5), \
    "single-head attention disagrees with PyTorch"
print("Basic checks passed — and one head matches PyTorch's kernel.")

### Coding Task 6.2 ★★ — Positional encoding, and what happens without it

Self-attention is **permutation equivariant**: shuffle the input
positions and the outputs shuffle identically. For a set of buses that
is a feature. For a time series it is fatal — a load profile read
backwards would produce the same forecast.

*Reuses what you wrote in 6.1 — do those first.*

**Your task.** Implement sinusoidal positional encoding, then demonstrate the
equivariance empirically: show that attention alone is blind to order
and that adding the encoding fixes it.

**Requirements**

- `positional_encoding(length, d_model)` returns `(length, d_model)`.
- Even indices use sine, odd indices cosine.
- Demonstrate the permutation property with an assertion, not prose.
- Plot the encoding as a heatmap.

<details><summary><b>Hints</b></summary>

- $PE_{(p, 2i)} = \sin(p / 10000^{2i/d})$ and $PE_{(p, 2i+1)} = \cos(p / 10000^{2i/d})$.
- Compute the divisor in log space — `torch.exp(arange(0, d, 2) * (-log(10000) / d))` — to avoid overflow at large `d_model`.
- To show equivariance, permute the input rows and compare against the same permutation of the output rows.

</details>

**Expected outcome.** The equivariance assertion passes without positional encoding and fails
with it. The heatmap shows smooth low-frequency stripes in the early
dimensions and fast oscillation in the later ones.

In [ ]:
import matplotlib.pyplot as plt

def positional_encoding(length, d_model):
    """Sinusoidal positional encoding of shape (length, d_model)."""
    position = torch.arange(length, dtype=torch.float32).unsqueeze(1)
    # TODO 1: the geometric frequency ladder, computed in log space
    divisor = ____
    encoding = torch.zeros(length, d_model)
    # TODO 2: sine into the even columns, cosine into the odd ones
    encoding[:, 0::2] = ____
    encoding[:, 1::2] = ____
    return encoding

pe = positional_encoding(CONTEXT, 64)
fig, ax = plt.subplots(figsize=(8, 3.2))
image = ax.imshow(pe.T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(image, ax=ax, label="value")
ax.set_xlabel("position (hour in the context window)")
ax.set_ylabel("embedding dimension")
ax.set_title("Sinusoidal positional encoding")
plt.show()

# TODO 3: show that bare attention cannot tell the order apart.
set_seed()
layer = MultiHeadSelfAttention(d_model=64, n_heads=4)
x = torch.randn(1, 10, 64)
perm = torch.randperm(10)

out_plain, _ = layer(x)
out_permuted, _ = layer(____)
assert torch.allclose(out_permuted, ____, atol=1e-5), \
    "attention should be permutation equivariant"
print("Without positional encoding, order carries no information.")

out_pe, _ = layer(x + pe[:10])
out_pe_permuted, _ = layer(x[:, perm] + pe[:10])
print(f"With positional encoding, the same test fails: "
      f"{not torch.allclose(out_pe_permuted, out_pe[:, perm], atol=1e-5)}")

In [ ]:
_pe = positional_encoding(50, 32)
assert _pe.shape == (50, 32)
assert _pe.abs().max() <= 1.0 + 1e-6, "sin and cos are bounded by 1"
assert not torch.allclose(_pe[0], _pe[1]), "positions must be distinguishable"
print("Basic checks passed.")

### Coding Task 6.3 ★★ — Assemble a pre-norm Transformer block

The block is `x + Attention(LN(x))` followed by `x + MLP(LN(x))`. Two
choices deserve naming:

- the **residual** makes the block an edit to the representation
  rather than a replacement, which is what lets dozens stack;
- **pre-norm** (normalise *before* the sublayer) trains without a
  learning-rate warm-up, unlike the post-norm arrangement in the 2017
  paper.

*Reuses what you wrote in 6.1 — do those first.*

**Your task.** Implement the block, then measure the difference the residual makes by
comparing gradient magnitudes at the first layer of a deep stack with
and without it.

**Requirements**

- `TransformerBlock(d_model, n_heads, d_ff=None, dropout=0.1)`, with `d_ff` defaulting to `4 * d_model`.
- Pre-norm ordering.
- The feed-forward network uses GELU.
- Compare gradient norms at layer 0 of an 8-block stack, residual on versus off.

<details><summary><b>Hints</b></summary>

- The feed-forward network is position-wise: it maps `d_model` -> `d_ff` -> `d_model` and sees each time step independently.
- To disable the residual, return the sublayer output directly instead of adding `x`.

</details>

**Expected outcome.** Without residuals the first block's gradient should be orders of
magnitude smaller. That is the vanishing-gradient problem of chapter 03
reappearing in depth rather than in time.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1, residual=True):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.residual = residual
        self.norm1 = nn.LayerNorm(d_model)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(d_model)
        # TODO 1: the position-wise feed-forward network
        self.feed_forward = nn.Sequential(
            ____,
            nn.GELU(),
            nn.Dropout(dropout),
            ____,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # TODO 2: pre-norm attention sublayer, then the MLP sublayer
        attended, weights = self.attention(____, mask=mask)
        x = ____ if self.residual else self.dropout(attended)
        fed = self.dropout(self.feed_forward(____))
        x = ____ if self.residual else fed
        return x, weights

def first_layer_gradient(residual, depth=8, d_model=64):
    set_seed()
    blocks = nn.ModuleList(
        TransformerBlock(d_model, 4, dropout=0.0, residual=residual)
        for _ in range(depth)
    )
    h = torch.randn(4, 16, d_model, requires_grad=True)
    x = h
    for block in blocks:
        x, _ = block(x)
    x.pow(2).mean().backward()
    return float(
        sum(p.grad.norm() for p in blocks[0].parameters() if p.grad is not None)
    )

with_residual = first_layer_gradient(residual=True)
without_residual = first_layer_gradient(residual=False)
print(f"gradient norm at block 0, with residuals   : {with_residual:.3e}")
print(f"gradient norm at block 0, without residuals: {without_residual:.3e}")
print(f"ratio: {with_residual / max(without_residual, 1e-30):.1f}x")

In [ ]:
_b = TransformerBlock(32, 4, dropout=0.0)
_x = torch.randn(2, 6, 32)
_y, _w = _b(_x)
assert _y.shape == _x.shape, "a block must preserve its input shape"
assert with_residual > without_residual, \
    "residual connections should preserve gradient magnitude"
print("Basic checks passed.")

### Analysis Task 6.4 ★★★ — Transformer versus LSTM on identical windows

The claim "Transformers beat RNNs" is about *scale and
parallelism*, not about small-data accuracy. At a few thousand training
windows on one series, the honest expectation is a tie.

**Your task.** Train a small Transformer forecaster on the same windows chapter 03
used, compare it against an LSTM under a matched optimisation budget,
and report both accuracy and wall-clock time.

**Requirements**

- Both models see identical data, identical scaling and the same number of epochs.
- Report test MAE in MW and training time for each.
- State a conclusion the numbers support — including if it is 'no meaningful difference'.

<details><summary><b>Hints</b></summary>

- Keep both small: `d_model=64`, 2 layers, 4 heads is plenty here.
- `train_model(model, train_loader, valid_loader, loss_fn, config)` returns a `History` with `best_val_loss` and `seconds`.

</details>

**Expected outcome.** Comparable accuracy. The Transformer's advantage in this regime is
*not* accuracy — say so plainly rather than manufacturing a winner.

In [ ]:
import pandas as pd

from ai_power_course.models.forecasters import LSTMForecaster, TransformerForecaster

EPOCHS = scaled(full=12, fast=2)
train_loader = make_loader(xs_train, ys_train, batch_size=64, shuffle=True)
valid_loader = make_loader(xs_valid, ys_valid, batch_size=256)

def fit_and_score(model, name):
    set_seed()
    history = train_model(
        model, train_loader, valid_loader, loss_fn=nn.MSELoss(),
        config=TrainConfig(epochs=EPOCHS, learning_rate=1e-3,
                           patience=4, verbose=False),
    )
    model.eval()
    with torch.no_grad():
        scaled_out = model(torch.tensor(xs_test, dtype=torch.float32)).numpy()
    prediction = y_scaler.inverse_transform(scaled_out)
    return {
        "model": name,
        "parameters": sum(p.numel() for p in model.parameters()),
        "MAE [MW]": mae(y_test.ravel(), prediction.ravel()),
        "train [s]": history.seconds,
    }

# TODO: build both models and score them
rows = [
    fit_and_score(____, "LSTM"),
    fit_and_score(____, "Transformer"),
]
comparison = pd.DataFrame(rows).set_index("model")
display(comparison.round(1))

In [ ]:
assert set(comparison.index) == {"LSTM", "Transformer"}
assert (comparison["MAE [MW]"] > 0).all()
assert comparison["MAE [MW]"].max() < 20000, "check the inverse transform"
print("Basic checks passed.")

---

## Exercise 07 — Language Models

*Builds on [`07_language_models.ipynb`](07_language_models.ipynb).*

Same architecture as chapter 06, one change: a causal mask and a
discrete vocabulary. The training signal — predict the next token — needs no
labels at all, which is what makes the internet a training set.

The corpus here is a synthetic grid-operations handbook. It is far too small
to produce a good model, and that is instructive: you will watch a model
memorise rather than generalise, and see exactly what scale buys.

In [ ]:
import numpy as np
import torch
from torch import nn

from ai_power_course.config import scaled, set_seed
from ai_power_course.corpus import HANDBOOK, generate_corpus
from ai_power_course.models.tinygpt import CharTokenizer, TinyGPT, TinyGPTConfig, generate

set_seed()
torch.set_num_threads(min(4, torch.get_num_threads()))

corpus = generate_corpus(n_logs=500, n_reports=140, n_assets=140)
print(f"corpus: {len(corpus):,} characters, {len(corpus.split()):,} words")
print(f"\n{corpus[:320]}...")

### Coding Task 7.1 ★★ — Tokenize, and build next-token batches

A language model sees integers. The tokenizer decides what an integer
means, and that choice determines the sequence length, the vocabulary
size and whether unseen words are even representable.

Character level keeps the vocabulary near 70 symbols so a tiny model
can learn something, and makes the step completely transparent.

**Your task.** Build a character tokenizer, encode the corpus, and write the batching
function that produces `(input, target)` pairs for next-token
prediction.

**Requirements**

- `encode` / `decode` must round-trip exactly.
- `get_batch(data, batch_size, context)` returns two tensors of shape `(batch_size, context)`.
- The target is the input shifted by exactly one position.
- Hold out the last 10% of the corpus for validation.

<details><summary><b>Hints</b></summary>

- For a start index `i`, the input is `data[i : i+context]` and the target is `data[i+1 : i+1+context]`.
- The largest legal start index is `len(data) - context - 1`.

</details>

**Expected outcome.** A vocabulary of roughly 70 characters, an exact round-trip, and a batch
whose target is visibly the input shifted left by one.

In [ ]:
tokenizer = CharTokenizer(corpus)
print(f"vocabulary: {tokenizer.vocab_size} characters")
print("".join(tokenizer.chars).replace("\n", "\\n"))

# TODO 1: encode the corpus and split off the last 10% for validation
data = torch.tensor(____, dtype=torch.long)
split_at = ____
train_data, valid_data = ____, ____
print(f"\ntrain {len(train_data):,} tokens | validation {len(valid_data):,} tokens")

CONTEXT = 128

def get_batch(data, batch_size=32, context=CONTEXT, generator=None):
    """Random windows and their one-step-shifted targets."""
    # TODO 2: draw `batch_size` random start indices, then stack
    starts = ____
    x = ____
    y = ____
    return x, y

gen = torch.Generator().manual_seed(0)
xb, yb = get_batch(train_data, batch_size=4, generator=gen)
print(f"\ninput {tuple(xb.shape)}   target {tuple(yb.shape)}")
print(f"\ninput  : {tokenizer.decode(xb[0][:60])!r}")
print(f"target : {tokenizer.decode(yb[0][:60])!r}")

In [ ]:
_text = "Bus voltage at Nordfeld fell to 0.94 per unit."
assert tokenizer.decode(tokenizer.encode(_text)) == _text, "round-trip failed"
assert xb.shape == yb.shape == (4, CONTEXT)
assert torch.equal(xb[0, 1:], yb[0, :-1]), "the target must be the input shifted by 1"
print("Basic checks passed.")

### Coding Task 7.2 ★★ — The loss, and what perplexity means

The loss is one line: cross-entropy between the logits at each
position and the token that actually followed.

Perplexity is $e^{\text{loss}}$, and it has a reading: *the effective
number of equally likely choices the model is deciding between*. A
uniform model over a 70-character vocabulary has perplexity 70.

*Reuses what you wrote in 7.1 — do those first.*

**Your task.** Instantiate a TinyGPT, compute its loss before any training, and check
that the perplexity matches the vocabulary size.

**Requirements**

- Predict the untrained loss from the vocabulary size **before** running the model.
- Verify the prediction to within 10%.
- Report the parameter count.

<details><summary><b>Hints</b></summary>

- An untrained model is roughly uniform over the vocabulary, so the cross-entropy should be near `log(vocab_size)`.
- `TinyGPT.forward(idx, targets)` returns `(logits, loss)`.

</details>

**Expected outcome.** A loss near `log(70) ≈ 4.25` and a perplexity near 70. If it is far off,
the initialisation or the target alignment is wrong.

In [ ]:
import math

# TODO 1: predict the untrained loss and perplexity from first principles
predicted_loss = ____
predicted_perplexity = ____
print(f"predicted untrained loss: {predicted_loss:.3f} "
      f"(perplexity {predicted_perplexity:.1f})")

set_seed()
config = TinyGPTConfig(
    vocab_size=tokenizer.vocab_size, context=CONTEXT,
    d_model=128, n_heads=4, n_layers=4, dropout=0.1,
)
model = TinyGPT(config)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

# TODO 2: measure it
xb, yb = get_batch(train_data, batch_size=16, generator=gen)
with torch.no_grad():
    logits, loss = ____
measured_perplexity = ____
print(f"measured untrained loss : {float(loss):.3f} "
      f"(perplexity {measured_perplexity:.1f})")

In [ ]:
assert abs(float(loss) - predicted_loss) < 0.1 * predicted_loss, \
    "an untrained model should be close to uniform"
assert logits.shape == (16, CONTEXT, tokenizer.vocab_size)
print("Basic checks passed.")

### Coding Task 7.3 ★★ — Train the model and watch it memorise

A few hundred thousand characters is not a corpus. The model will fit
it and then start reciting it, and seeing that happen is more useful
than any explanation of why scale matters.

*Reuses what you wrote in 7.2 — do those first.*

**Your task.** Write the training loop, track the training and validation loss, and
generate a sample from the model before and after.

**Requirements**

- Estimate the validation loss periodically under `torch.no_grad()`.
- Clip gradients to a maximum norm of 1.0.
- Generate a sample from an untrained and a trained model, and show both.

<details><summary><b>Hints</b></summary>

- `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` goes between `backward()` and `step()`.
- `generate(model, idx, max_new_tokens=..., temperature=...)` expects `idx` of shape `(batch, length)`.

</details>

**Expected outcome.** The training loss should fall well below the validation loss — that gap
*is* the memorisation. The generated text should acquire the shape of
the corpus (substation names, per-unit voltages, MW figures) without
necessarily meaning anything.

In [ ]:
prompt = torch.tensor([tokenizer.encode("Bus voltage at ")], dtype=torch.long)
before = generate(model, prompt, max_new_tokens=120, temperature=0.8,
                  generator=torch.Generator().manual_seed(0))
print("BEFORE TRAINING")
print(tokenizer.decode(before[0]))

STEPS = scaled(full=1200, fast=60)
EVAL_EVERY = max(1, STEPS // 8)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
gen = torch.Generator().manual_seed(1)

@torch.no_grad()
def estimate_loss(data, batches=8):
    model.eval()
    losses = []
    for _ in range(batches):
        x, y = get_batch(data, batch_size=16, generator=gen)
        losses.append(float(model(x, y)[1]))
    model.train()
    return float(np.mean(losses))

loss_log = []
for step in range(1, STEPS + 1):
    x, y = get_batch(train_data, batch_size=16, generator=gen)
    # TODO: forward, backward with gradient clipping, step
    _logits, loss = ____
    ____
    ____
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    ____
    if step % EVAL_EVERY == 0 or step == 1:
        row = {"step": step, "train": estimate_loss(train_data),
               "validation": estimate_loss(valid_data)}
        loss_log.append(row)
        print(f"step {step:5d}  train {row['train']:.3f}  "
              f"validation {row['validation']:.3f}")

after = generate(model, prompt, max_new_tokens=200, temperature=0.8,
                 generator=torch.Generator().manual_seed(0))
print("\nAFTER TRAINING")
print(tokenizer.decode(after[0]))

In [ ]:
assert loss_log[-1]["train"] < loss_log[0]["train"], "the training loss did not fall"
assert loss_log[-1]["train"] < math.log(tokenizer.vocab_size), \
    "the model should beat a uniform distribution"
print("Basic checks passed.")

### Coding Task 7.4 ★★ — Temperature and top-k sampling

The model outputs a distribution. Turning it into text is a separate
decision, and it has as much effect on what you read as the weights do.

Temperature rescales the logits before the softmax: below 1 sharpens,
above 1 flattens, 0 is greedy. Top-k truncates to the `k` most likely
tokens.

*Reuses what you wrote in 7.3 — do those first.*

**Your task.** Generate from the trained model at several temperatures and with top-k
truncation, then quantify the diversity of each setting rather than
judging by eye.

**Requirements**

- At least four settings, including greedy (`temperature=0`).
- Report a numeric diversity measure per setting, not only the text.
- Use the same seed for every setting so the comparison is fair.

<details><summary><b>Hints</b></summary>

- The fraction of unique 5-character sequences is a reasonable diversity proxy.
- `generate(..., top_k=20)` truncates the distribution to the 20 most likely next characters at each step.

</details>

**Expected outcome.** Greedy decoding should loop or repeat. High temperature should be
diverse and less coherent. The interesting question is where the
trade-off sits, not which end is 'right'.

In [ ]:
import pandas as pd

def diversity(text, n=5):
    """Fraction of character n-grams that are unique."""
    grams = [text[i : i + n] for i in range(len(text) - n)]
    return len(set(grams)) / max(len(grams), 1)

settings = [
    {"temperature": 0.0, "top_k": None, "label": "greedy (T=0)"},
    {"temperature": 0.5, "top_k": None, "label": "T=0.5"},
    {"temperature": 1.0, "top_k": None, "label": "T=1.0"},
    {"temperature": 1.5, "top_k": None, "label": "T=1.5"},
    {"temperature": 1.0, "top_k": 10, "label": "T=1.0, top-k=10"},
]

rows = []
for setting in settings:
    # TODO: generate 220 tokens with these settings, seeded identically
    sample = ____
    text = tokenizer.decode(sample[0])
    rows.append({"setting": setting["label"], "diversity": diversity(text)})
    print(f"--- {setting['label']} ---")
    print(text[:200].replace("\n", " | "))
    print()

display(pd.DataFrame(rows).set_index("setting").round(3))

In [ ]:
assert len(rows) == len(settings)
assert table.loc["greedy (T=0)", "diversity"] <= table.loc["T=1.5", "diversity"], \
    "greedy decoding should not be more diverse than T=1.5"
print("Basic checks passed.")

### Reflection Question 7.5 ★★★ — Is next-token prediction 'just autocomplete'?

The dismissal is common and it is not stupid: the training objective
genuinely is nothing more than predicting the next token. The question
is what that objective forces a model to represent.

**Your task.** Make the strongest case you can for *both* readings. Then take a
position: for a grid operations assistant, which reading should
determine how much you trust the system, and what would you measure to
decide?

**Expected outcome.** A good answer takes the deflationary reading seriously instead of
dismissing it, and lands on something measurable rather than on a
definition.

**The case for 'just autocomplete'.** …

**The case against.** …

**My position, and what I would measure.** …

---

## Exercise 08 — Pretrained LLMs and Adaptation

*Builds on [`08_pretrained_llms_and_adaptation.ipynb`](08_pretrained_llms_and_adaptation.ipynb).*

You stop training models and start adapting one. The spectrum runs from
zero-cost (embeddings, prompting) through cheap (linear probe, LoRA) to
expensive (full fine-tuning), and the engineering skill is choosing the
cheapest point that solves the problem.

The task: classify grid operations log entries into `routine`, `overload`,
`voltage_violation` and `outage`.

**Requires a network connection** the first time, to download a 23 M-parameter
encoder. Under `AI_POWER_COURSE_OFFLINE=1` these cells report and skip.

In [ ]:
import warnings

import numpy as np
import torch
from sklearn.model_selection import train_test_split

from ai_power_course.config import offline_mode, scaled, set_seed
from ai_power_course.corpus import EVENT_CLASSES, generate_event_dataset
from ai_power_course.metrics import classification_metrics

warnings.filterwarnings("ignore")
set_seed()
torch.set_num_threads(min(4, torch.get_num_threads()))

ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # 22.7 M params, Apache-2.0

texts, labels = generate_event_dataset(n_per_class=scaled(full=220, fast=60))
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.3, random_state=0, stratify=labels
)
print(f"{len(texts)} log entries across {len(EVENT_CLASSES)} classes: "
      f"{', '.join(EVENT_CLASSES)}")
print(f"train {len(train_texts)} | test {len(test_texts)}")

tokenizer = encoder = None
if offline_mode():
    print("\nAI_POWER_COURSE_OFFLINE is set — the download-dependent tasks will skip.")
else:
    import transformers
    from transformers import AutoModel, AutoTokenizer

    transformers.logging.set_verbosity_error()
    tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
    encoder = AutoModel.from_pretrained(ENCODER_NAME).eval()
    n_params = sum(p.numel() for p in encoder.parameters())
    print(f"\n{ENCODER_NAME}: {n_params:,} parameters, "
          f"{encoder.config.num_hidden_layers} layers, "
          f"hidden {encoder.config.hidden_size}")
    print("That is the chapter-06 architecture with weights someone else paid for.")

### Coding Task 8.1 ★★ — Mean pooling that respects the attention mask

An encoder returns one vector per token. To get one vector per
sentence you average them — but a batch is padded to its longest
member, and averaging the padding in dilutes every short sentence by a
different amount.

This is the single most common bug in home-made embedding code, and it
is invisible: the shapes are right and the numbers look reasonable.

**Your task.** Implement mask-aware mean pooling with L2 normalisation, then
demonstrate the bug by comparing against the naive version.

**Requirements**

- `embed(texts)` returns `(n_texts, 384)`, L2-normalised.
- Run under `torch.no_grad()` — nothing is being trained.
- Show numerically that naive pooling changes a sentence's embedding depending on what it was batched with.

<details><summary><b>Hints</b></summary>

- `tokenizer(texts, padding=True, truncation=True, return_tensors='pt')` gives `input_ids` and `attention_mask`.
- Expand the mask to the hidden width, multiply, sum over the token axis, and divide by the mask's sum — clamped away from zero.

</details>

**Expected outcome.** Correct pooling gives an identical embedding for a sentence regardless
of its batch. Naive pooling does not, and the difference should be large
enough to matter.

In [ ]:
if encoder is None:
    print("Skipped: offline mode.")
else:
    @torch.no_grad()
    def embed(texts, batch_size=64, masked=True):
        """Mean-pooled, L2-normalised sentence embeddings."""
        out = []
        for start in range(0, len(texts), batch_size):
            chunk = list(texts[start : start + batch_size])
            encoded = tokenizer(chunk, padding=True, truncation=True,
                                max_length=128, return_tensors="pt")
            hidden = encoder(**encoded).last_hidden_state
            if masked:
                # TODO 1: mask-aware mean over the token axis
                mask = encoded["attention_mask"].unsqueeze(-1).to(hidden.dtype)
                pooled = ____
            else:
                pooled = hidden.mean(dim=1)   # the bug, for comparison
            # TODO 2: L2-normalise
            out.append(____)
        return torch.cat(out).numpy()

    probe = "Scheduled inspection completed on the overhead line at Nordfeld."
    padded_batch = [probe, "A" * 400]        # forces heavy padding
    alone = embed([probe])[0]
    batched_ok = embed(padded_batch, masked=True)[0]
    batched_bad = embed(padded_batch, masked=False)[0]

    alone_bad = embed([probe], masked=False)[0]
    print(f"mask-aware, alone vs batched : "
          f"max difference {np.abs(alone - batched_ok).max():.2e}")
    print(f"naive,      alone vs batched : "
          f"max difference {np.abs(alone_bad - batched_bad).max():.2e}")

    X_train = embed(train_texts)
    X_test = embed(test_texts)
    print(f"\nembeddings: {X_train.shape}")

In [ ]:
if encoder is None:
    print("Skipped: offline mode.")
else:
    assert X_train.shape[1] == encoder.config.hidden_size
    assert np.allclose(np.linalg.norm(X_train, axis=1), 1.0, atol=1e-5), \
        "embeddings should be L2-normalised"
    assert np.abs(alone - batched_ok).max() < 1e-5, \
        "mask-aware pooling must be batch-independent"
    print("Basic checks passed.")

### Coding Task 8.2 ★★ — A linear probe on frozen embeddings

The cheapest useful adaptation: freeze the 23 M-parameter encoder,
fit a logistic regression on its output. Seconds to train, no
gradients through the backbone, and the same embeddings serve any
number of tasks.

*Reuses what you wrote in 8.1 — do those first.*

**Your task.** Fit a linear probe on the frozen embeddings, evaluate it, and compare
against a bag-of-words baseline on the same split.

**Requirements**

- Use `LogisticRegression` on the embeddings from task 8.1.
- Include a `TfidfVectorizer` + logistic regression baseline.
- Report accuracy and macro-F1 for both. `classification_metrics` is binary one-vs-rest, so macro-average it yourself.
- Report how many parameters each approach actually fits.

<details><summary><b>Hints</b></summary>

- `classification_metrics(y, p, positive_label=k)` scores class `k` against all the others and returns `F1` among its keys.
- The probe fits `384 * 4 + 4` parameters. The backbone's 22.7 M are frozen.

</details>

**Expected outcome.** Both should do well — this is an easy, templated task. Note honestly
if TF-IDF matches the embeddings, and think about what that says.

In [ ]:
if encoder is None:
    print("Skipped: offline mode.")
else:
    import pandas as pd
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression

    # TODO 1: the linear probe on frozen embeddings
    probe_model = ____
    probe_model.fit(____, ____)
    probe_pred = ____

    # TODO 2: a bag-of-words baseline on the same split
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    tfidf_train = vectorizer.fit_transform(train_texts)
    tfidf_test = ____
    tfidf_model = LogisticRegression(max_iter=1000).fit(tfidf_train, train_labels)
    tfidf_pred = ____

    def multiclass_scores(y_true, y_pred):
        """Accuracy, and macro-F1 as the unweighted mean of per-class F1.

        `classification_metrics` is binary one-vs-rest, so macro-averaging
        means running it once per class and taking the plain mean — which
        is exactly what "macro" means, and why it is the right metric when
        the rare classes are the ones you care about.
        """
        # TODO 3: run `classification_metrics` once per class, then
        #         average the F1 scores. Accuracy is the plain match rate.
        raise NotImplementedError

    rows = []
    for name, pred, n_fitted in [
        ("linear probe on MiniLM", probe_pred,
         X_train.shape[1] * len(EVENT_CLASSES) + len(EVENT_CLASSES)),
        ("TF-IDF + logistic regression", tfidf_pred,
         tfidf_train.shape[1] * len(EVENT_CLASSES) + len(EVENT_CLASSES)),
    ]:
        rows.append({"method": name, **multiclass_scores(test_labels, pred),
                     "fitted params": n_fitted})
    probe_table = pd.DataFrame(rows).set_index("method")
    display(probe_table.round(3))

In [ ]:
if encoder is None:
    print("Skipped: offline mode.")
else:
    assert probe_table["accuracy"].min() > 0.7, \
        "both methods should do well on this templated task"
    assert len(probe_pred) == len(test_labels)
    print("Basic checks passed.")

### Coding Task 8.3 ★★★ — LoRA: predict the parameter count, then verify it

LoRA freezes $W_0$ and learns a low-rank correction
$\Delta W = BA$ with $B \in \mathbb{R}^{d \times r}$,
$A \in \mathbb{R}^{r \times k}$ and $r \ll \min(d, k)$.

A $384 \times 384$ attention projection has 147,456 parameters. Its
rank-8 correction has $2 \times 384 \times 8 = 6{,}144$.

**Your task.** Work out from the architecture how many trainable parameters a LoRA
configuration will have, then build it with `peft` and check your
arithmetic against the real model.

**Requirements**

- Derive the count by hand **before** building the model.
- Target the `query` and `value` projections with `r=8`.
- Assert that your formula matches the actual trainable count.
- Report the trainable fraction of the whole model.

<details><summary><b>Hints</b></summary>

- Per targeted module: `2 * hidden * r` (the `A` and `B` matrices).
- Multiply by the number of targeted modules per layer and by the number of layers.
- `sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and 'lora' in n)` counts what was actually added.

</details>

**Expected outcome.** Your formula should match exactly. A trainable fraction well under 1%.

In [ ]:
if encoder is None:
    print("Skipped: offline mode.")
else:
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import AutoModelForSequenceClassification

    hidden = encoder.config.hidden_size
    n_layers = encoder.config.num_hidden_layers
    RANK = 8
    TARGETS = ["query", "value"]

    # TODO 1: predict the trainable LoRA parameter count from the architecture
    expected_lora = ____
    print(f"{n_layers} layers x {len(TARGETS)} modules x 2 matrices x "
          f"{hidden} x {RANK} = {expected_lora:,} expected")

    set_seed()
    base = AutoModelForSequenceClassification.from_pretrained(
        ENCODER_NAME, num_labels=len(EVENT_CLASSES)
    )
    # TODO 2: wrap it with LoRA
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=____,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=____,
    )
    lora_model = get_peft_model(base, lora_config)

    actual_lora = sum(p.numel() for n, p in lora_model.named_parameters()
                      if p.requires_grad and "lora" in n)
    total = sum(p.numel() for p in lora_model.parameters())
    trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
    print(f"actual LoRA parameters: {actual_lora:,}")
    print(f"trainable overall     : {trainable:,} of {total:,} "
          f"({trainable / total:.2%}) — the rest is the classifier head")

In [ ]:
if encoder is None:
    print("Skipped: offline mode.")
else:
    assert expected_lora == actual_lora, (
        f"formula gives {expected_lora:,} but the model has {actual_lora:,}"
    )
    assert trainable / total < 0.05, "LoRA should train a small fraction"
    print("Basic checks passed — the formula matches the model exactly.")

### Reflection Question 8.4 ★★ — Which adaptation method, and why?

The spectrum: embeddings → prompting → in-context learning → RAG →
linear probe → LoRA → full fine-tuning. Cost, data requirements and
achievable accuracy all increase along it, and so does the maintenance
burden.

**Your task.** For each of the four scenarios below, pick a method and justify it in
two or three sentences. Name the deciding constraint in each case.

1. 300 labelled operator log entries; a classifier is needed next week.
2. 50,000 labelled entries; accuracy is worth real money; a GPU is
   available.
3. No labels; engineers want to search 20 years of maintenance reports
   by meaning, not keyword.
4. The model must answer questions about internal grid codes that are
   updated quarterly.

**Expected outcome.** Case 4 is the one that separates answers. If your instinct is to
fine-tune on the documents, argue against yourself before committing.

| Scenario | Method | Deciding constraint |
| --- | --- | --- |
| 1. 300 labels, one week | … | … |
| 2. 50,000 labels, GPU | … | … |
| 3. No labels, semantic search | … | … |
| 4. Quarterly-updated grid codes | … | … |

**Reasoning.**

1. …
2. …
3. …
4. …

---

## Exercise 09 — Foundation Models Beyond LLMs

*Builds on [`09_foundation_models_beyond_llms.ipynb`](09_foundation_models_beyond_llms.ipynb).*

The chapter that makes the course's argument concrete. A time-series
foundation model has no vocabulary, no tokens in the linguistic sense and
generates nothing you would call text — and it is unmistakably a foundation
model: pretrained on a broad distribution, adapted to many downstream tasks
without task-specific training.

You will run one zero-shot against specialists trained on the target data.

**Requires a network connection** the first time. Under
`AI_POWER_COURSE_OFFLINE=1` the foundation-model cells report and skip, and
the rest still runs.

In [ ]:
import numpy as np
import pandas as pd
import torch

from ai_power_course.config import fast_mode, offline_mode, scaled, set_seed
from ai_power_course.data import load_energy_data, time_split
from ai_power_course.metrics import coverage, crps_from_quantiles, mae, pinball_loss

set_seed()
torch.set_num_threads(min(4, torch.get_num_threads()))

dataset = load_energy_data()
frame = dataset.frame
train_raw, valid_raw, test_raw = time_split(frame)

CONTEXT, HORIZON = 512, 24
QUANTILES = [0.1, 0.5, 0.9]

# One forecast per day, issued at 00:00 UTC — a realistic day-ahead cadence.
origins = test_raw.index[test_raw.index.hour == 0]
origins = origins[origins >= frame.index[0] + pd.Timedelta(hours=CONTEXT)]
origins = origins[origins + pd.Timedelta(hours=HORIZON) <= frame.index[-1]]
origins = origins[: scaled(full=120, fast=12)]

series = frame.load_mw
positions = series.index.get_indexer(origins)
contexts = np.stack([series.to_numpy()[p - CONTEXT + 1 : p + 1] for p in positions])
targets = np.stack([series.to_numpy()[p + 1 : p + 1 + HORIZON] for p in positions])
print(f"{len(origins)} daily forecast origins, "
      f"{origins[0]:%Y-%m-%d} to {origins[-1]:%Y-%m-%d}")
print(f"contexts {contexts.shape}   targets {targets.shape}")
print(f"\n{dataset.describe()}")

### Coding Task 9.1 ★★ — A shared evaluation harness

Before any model is loaded, fix the protocol. Every method will be
scored on exactly the same target values, with the same metric, against
the same reference — otherwise the comparison measures the protocol
rather than the models.

**Your task.** Build the baseline forecasts and a scoring function that reports MAE
and skill against the seasonal naive.

**Requirements**

- A flat persistence baseline and a 168-hour seasonal naive.
- `score(name, prediction)` records into a shared dictionary.
- Skill is measured against the seasonal naive, and the reference must therefore score exactly 0.
- Every prediction has shape `(n_origins, HORIZON)`.

<details><summary><b>Hints</b></summary>

- Persistence at all 24 lead times is the value at the origin, repeated: `np.repeat(contexts[:, -1:], HORIZON, axis=1)`.
- Skill = `1 - MAE_model / MAE_reference`.

</details>

**Expected outcome.** The seasonal naive should clearly beat flat persistence, and scoring the
reference against itself must give exactly zero skill.

In [ ]:
predictions = {}
scores = {}

# TODO 1: the two baselines
predictions["Persistence (flat)"] = ____
predictions["Seasonal naive (168 h)"] = np.stack(
    [series.to_numpy()[p + 1 - 168 : p + 1 - 168 + HORIZON] for p in positions]
)

REFERENCE = "Seasonal naive (168 h)"

def score(name, prediction=None):
    """Record MAE and skill against the reference forecast."""
    prediction = predictions[name] if prediction is None else prediction
    if prediction.shape != targets.shape:
        raise ValueError(f"{name}: expected {targets.shape}, got {prediction.shape}")
    # TODO 2: MAE over all origins and lead times, then skill
    model_mae = ____
    reference_mae = ____
    scores[name] = {"MAE [MW]": model_mae,
                    "skill vs naive": ____}
    return scores[name]

for name in predictions:
    result = score(name)
    print(f"{name:26s} MAE {result['MAE [MW]']:8,.1f} MW   "
          f"skill {result['skill vs naive']:+.3f}")

In [ ]:
assert abs(scores[REFERENCE]["skill vs naive"]) < 1e-12, \
    "the reference must score exactly zero skill against itself"
assert scores[REFERENCE]["MAE [MW]"] < scores["Persistence (flat)"]["MAE [MW]"], \
    "the weekly naive should beat flat persistence at a 24 h horizon"
for _name, _p in predictions.items():
    assert _p.shape == targets.shape, f"{_name} has the wrong shape"
print("Basic checks passed.")

### Coding Task 9.2 ★★★ — Run a real foundation model zero-shot

Chronos-2 (Ansari et al., 2025) is a 120 M-parameter encoder-only model
pretrained on a broad corpus of time series. It has never seen this
dataset — it is synthetic, generated for this course — so contamination
is genuinely ruled out here in a way it almost never is on a public
benchmark.

No training. No fine-tuning. Hand it 512 numbers, ask for 24.

*Reuses what you wrote in 9.1 — do those first.*

**Your task.** Load Chronos-2, produce quantile forecasts for every origin, and score
it in the harness from task 9.1 alongside a specialist trained on this
data.

**Requirements**

- Guard the whole block so it degrades gracefully offline.
- Request the quantile levels fixed in `QUANTILES`.
- Report inference time per series.
- Fit one specialist — gradient boosting, one model per lead time — on the training split, and score it in the same harness.

<details><summary><b>Hints</b></summary>

- `Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map='cpu')`.
- `predict_quantiles(list_of_tensors, prediction_length=..., quantile_levels=...)` returns `(quantiles, mean)`; each quantile element is `(n_variates, horizon, n_quantiles)`, so index `[0]` for a univariate series.
- The median is `QUANTILES.index(0.5)` along the last axis.

</details>

**Expected outcome.** A zero-shot model that is competitive with, and may beat, a specialist
trained on this exact series. Whichever way it lands, report it — the
interesting content is the comparison, not a win.

In [ ]:
import time

chronos_quantiles = None
if offline_mode():
    print("Skipped: offline mode.")
else:
    try:
        from chronos import Chronos2Pipeline

        started = time.perf_counter()
        pipeline = Chronos2Pipeline.from_pretrained(
            "amazon/chronos-2", device_map="cpu")
        n_params = sum(p.numel() for p in pipeline.model.parameters())
        print(f"loaded {n_params / 1e6:.0f} M parameters in "
              f"{time.perf_counter() - started:.0f}s")

        started = time.perf_counter()
        # TODO 1: forecast every context window
        quantile_list, _mean = pipeline.predict_quantiles(
            ____,
            prediction_length=____,
            quantile_levels=____,
            batch_size=16,
        )
        elapsed = time.perf_counter() - started
        chronos_quantiles = torch.stack([q[0] for q in quantile_list]).numpy()
        # TODO 2: the median forecast
        predictions["Chronos-2 (zero-shot)"] = ____
        print(f"{len(contexts)} series in {elapsed:.0f}s "
              f"({elapsed / len(contexts) * 1000:.0f} ms each)")
        print(score("Chronos-2 (zero-shot)"))
    except Exception as exc:
        print(f"Chronos-2 unavailable ({type(exc).__name__}: {str(exc)[:160]})")

# A specialist trained on THIS series, as the fair point of comparison.
from sklearn.ensemble import HistGradientBoostingRegressor

from ai_power_course.data import make_supervised

specialist = np.zeros_like(targets)
for lead in range(HORIZON):
    X_fit, y_fit = make_supervised(train_raw, horizon=lead + 1,
                                   exogenous=("temperature_c",))
    model = HistGradientBoostingRegressor(max_iter=scaled(full=120, fast=30),
                                          random_state=0).fit(X_fit, y_fit)
    X_eval, _ = make_supervised(frame, horizon=lead + 1,
                                exogenous=("temperature_c",))
    specialist[:, lead] = model.predict(X_eval.loc[origins])
predictions["Gradient boosting (specialist)"] = specialist
print(score("Gradient boosting (specialist)"))

In [ ]:
assert "Gradient boosting (specialist)" in scores
assert scores["Gradient boosting (specialist)"]["skill vs naive"] > 0, \
    "a trained specialist should beat the naive baseline"
if chronos_quantiles is not None:
    assert chronos_quantiles.shape == (len(origins), HORIZON, len(QUANTILES))
    _q = chronos_quantiles
    assert (_q[:, :, 0] <= _q[:, :, 2] + 1e-6).all(), \
        "the 10% quantile must not exceed the 90% quantile"
print("Basic checks passed.")

### Coding Task 9.3 ★★★ — Probabilistic evaluation: sharpness and calibration

Grid operations decisions are about risk, not point estimates. Reserve
procurement asks "how bad could the peak be?", not "what is the
expected peak?".

A point metric cannot answer that, and it cannot distinguish a
well-calibrated wide interval from an overconfident narrow one.

*Reuses what you wrote in 9.2 — do those first.*

**Your task.** Score the Chronos-2 quantile forecasts with the pinball loss, CRPS,
and empirical coverage of the 80% interval, and plot one day with the
interval shaded.

**Requirements**

- Pinball loss at each of the three quantile levels.
- CRPS approximated from the quantiles.
- Empirical coverage of the [0.1, 0.9] interval, compared with the nominal 80%.
- One plot showing the context, the truth, the median and the band.

<details><summary><b>Hints</b></summary>

- `pinball_loss(y_true, y_quantile, quantile)`, `crps_from_quantiles(y_true, quantiles, levels)` and `coverage(y_true, lower, upper)` are already imported.
- Coverage far below nominal means overconfidence — the interval is too narrow for the errors the model actually makes.

</details>

**Expected outcome.** Coverage somewhere near 80%. Say which direction it misses in: below
nominal is overconfident, above is conservative, and they have opposite
operational consequences.

In [ ]:
import matplotlib.pyplot as plt

if chronos_quantiles is None:
    print("Skipped: no quantile forecasts available (offline mode).")
else:
    flat_truth = targets.ravel()
    # TODO 1: pinball loss at each level
    for i, level in enumerate(QUANTILES):
        loss = ____
        print(f"pinball loss at q{level:.1f}: {loss:8,.1f} MW")

    # TODO 2: CRPS and coverage of the 80% interval
    crps = ____
    lower = chronos_quantiles[:, :, QUANTILES.index(0.1)].ravel()
    upper = chronos_quantiles[:, :, QUANTILES.index(0.9)].ravel()
    empirical = ____
    print(f"\nCRPS                     : {crps:8,.1f} MW")
    print(f"80% interval coverage    : {empirical:.1%} (nominal 80.0%)")
    print(f"mean interval width      : {np.mean(upper - lower):8,.1f} MW")

    day = 0
    hours = np.arange(HORIZON)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(np.arange(-48, 0), contexts[day, -48:], color="0.5", label="context")
    ax.plot(hours, targets[day], "o-", color="black", ms=3, label="actual")
    ax.plot(hours, chronos_quantiles[day, :, 1], "s-", ms=3,
            label="Chronos-2 median")
    ax.fill_between(hours, chronos_quantiles[day, :, 0],
                    chronos_quantiles[day, :, 2], alpha=0.25,
                    label="80% interval")
    ax.axvline(-0.5, color="0.7", lw=1)
    ax.set_xlabel("hours from the forecast origin")
    ax.set_ylabel("load [MW]")
    ax.set_title(f"Zero-shot day-ahead forecast, {origins[day]:%Y-%m-%d}")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    plt.show()

In [ ]:
if chronos_quantiles is None:
    print("Skipped: offline mode.")
else:
    assert 0.0 <= empirical <= 1.0
    assert crps > 0, "CRPS is a positive loss"
    assert (upper >= lower - 1e-6).all(), "the interval bounds are crossed"
    print("Basic checks passed.")

### Reflection Question 9.4 ★★★ — What does a foundation model actually change?

The measured result is likely to be undramatic: a zero-shot model
roughly level with a specialist trained on the target data. It is worth
being precise about why that would still be significant, and about what
it would not establish.

**Your task.** Answer three questions. (a) If the zero-shot model merely ties the
specialist, what has actually changed for a practitioner? (b) Name two
situations where you would still choose the specialist. (c) What claim
would this experiment *not* support, even if the foundation model won
outright?

**Expected outcome.** Part (c) is the one that matters. A single-series, single-horizon
comparison on synthetic data supports a narrow claim, and naming its
limits precisely is the skill being tested.

**(a) What changes if it only ties.** …

**(b) Two cases for the specialist.** …

**(c) What this experiment does not support.** …

---

## Exercise 10 — Towards Grid Foundation Models

*Builds on [`10_grid_foundation_models.ipynb`](10_grid_foundation_models.ipynb).*

Everything converges here. A power grid is not a sequence and not an image:
it is a graph with physics on it, a different graph for every network, and
bus numbering that carries no meaning.

You will build a small grid foundation model — pretrain a message-passing
encoder on a population of networks with a masked objective, then transfer it
to a grid it has never seen — and validate it against the physics, not only
against the statistics.

This is the frontier. Nothing in this chapter is settled.

In [ ]:
import numpy as np
import torch
from torch import nn

from ai_power_course.config import fast_mode, scaled, set_seed
from ai_power_course.grid.graphs import (
    EDGE_FEATURE_NAMES, NODE_FEATURE_NAMES, STATE_FEATURE_NAMES, net_to_graph,
)
from ai_power_course.grid.networks import load_network, run_power_flow
from ai_power_course.grid.sampling import SamplingConfig, sample_operating_points
from ai_power_course.metrics import mae
from ai_power_course.models.gnn import GraphBatch, MaskedGridModel, collate_graphs

set_seed()
torch.set_num_threads(min(4, torch.get_num_threads()))

PRETRAIN = ("case9", "case14", "case30")
HELDOUT = "case33bw"
N_SAMPLES = scaled(full=80, fast=26)

print(f"node features ({len(NODE_FEATURE_NAMES)}): {', '.join(NODE_FEATURE_NAMES)}")
print(f"  maskable state: {', '.join(STATE_FEATURE_NAMES)}")
print(f"edge features ({len(EDGE_FEATURE_NAMES)}): {', '.join(EDGE_FEATURE_NAMES)}")

config = SamplingConfig(n_samples=N_SAMPLES, seed=20260101)
pretrain_graphs, heldout_graphs = [], []
for offset, name in enumerate(PRETRAIN):
    points, _ = sample_operating_points(
        name, SamplingConfig(**{**config.__dict__, "seed": config.seed + 100 * offset})
    )
    pretrain_graphs.extend(p.graph for p in points)
points, _ = sample_operating_points(
    HELDOUT, SamplingConfig(**{**config.__dict__, "seed": config.seed + 9000})
)
heldout_graphs = [p.graph for p in points]

print(f"\npretraining: {len(pretrain_graphs)} states over {len(PRETRAIN)} grids "
      f"({', '.join(PRETRAIN)})")
print(f"held out   : {len(heldout_graphs)} states of {HELDOUT}, "
      f"{heldout_graphs[0]['node_features'].shape[0]} buses — never seen in pretraining")

### Coding Task 10.1 ★★★ — Permutation equivariance is not optional

Bus 14 is bus 14 because someone typed it. Renumber the network and it
is the same grid, so the model's per-bus outputs must simply come back
in the new order.

An MLP on a flattened state vector fails this immediately. A
message-passing network satisfies it by construction — and it is worth
*testing* rather than assuming, because a single indexing mistake breaks
it silently.

**Your task.** Write a function that permutes a graph's node ordering consistently,
then verify that the encoder's output permutes identically.

**Requirements**

- `permute_graph(graph, perm)` relabels the nodes *and* the edge endpoints.
- Edge features must follow their edges unchanged.
- Verify equivariance to `1e-5`.
- Also show that an MLP on the flattened state is **not** equivariant.

<details><summary><b>Hints</b></summary>

- If `perm` gives the new position of each old node, then `inverse[perm] = arange(n)` maps old indices to new ones, and `node_features[perm]` reorders the rows.
- Edges are indices into the node array, so apply the inverse mapping to `edge_index`.

</details>

**Expected outcome.** The message-passing encoder should match to floating-point precision.
The MLP should not — by a wide margin.

In [ ]:
def permute_graph(graph, perm):
    """Relabel the buses of a graph. Same grid, different numbering."""
    n = graph["node_features"].shape[0]
    inverse = torch.empty(n, dtype=torch.long)
    inverse[perm] = torch.arange(n)
    return {
        # TODO 1: reorder the node rows
        "node_features": ____,
        # TODO 2: remap the edge endpoints
        "edge_index": ____,
        "edge_features": graph["edge_features"],
    }

set_seed()
model = MaskedGridModel(
    node_dim=len(NODE_FEATURE_NAMES),
    edge_dim=len(EDGE_FEATURE_NAMES),
    d_model=64, n_layers=4, n_state=len(STATE_FEATURE_NAMES),
)

graph = heldout_graphs[0]
n_buses = graph["node_features"].shape[0]
perm = torch.randperm(n_buses, generator=torch.Generator().manual_seed(0))

original = model.embed(collate_graphs([graph]))
permuted = model.embed(collate_graphs([permute_graph(graph, perm)]))

# TODO 3: the permuted embedding should equal the permuted original
gap = float((permuted - ____).abs().max())
print(f"message passing, max |difference|: {gap:.2e}")
assert gap < 1e-5, "the grid encoder must be permutation equivariant"

# The contrast: an MLP on the flattened state vector.
set_seed()
flat_mlp = nn.Sequential(
    nn.Linear(n_buses * len(NODE_FEATURE_NAMES), 64), nn.GELU(), nn.Linear(64, 64)
)
flat_original = flat_mlp(graph["node_features"].reshape(1, -1))
flat_permuted = flat_mlp(graph["node_features"][perm].reshape(1, -1))
print(f"flat MLP,         max |difference|: "
      f"{float((flat_permuted - flat_original).abs().max()):.2e}")

In [ ]:
_g = permute_graph(heldout_graphs[1], torch.arange(
    heldout_graphs[1]["node_features"].shape[0]))
assert torch.equal(_g["edge_index"], heldout_graphs[1]["edge_index"]), \
    "the identity permutation must leave the graph unchanged"
assert gap < 1e-5
print("Basic checks passed.")

### Coding Task 10.2 ★★★ — Message passing in ten lines

Every graph neural network is the same three steps: compute a message
along each edge, sum the messages arriving at each node, update the node
from its own state and that sum.

$$m_{i \to j} = \phi([h_i, h_j, e_{ij}]), \qquad
  h_j' = h_j + \psi([h_j, \textstyle\sum_{i \in N(j)} m_{i \to j}])$$

No library needed. `index_add_` does the aggregation.

**Your task.** Implement one message-passing layer from scratch and verify that the
aggregation matches an explicit per-node loop.

**Requirements**

- `forward(x, edge_index, edge_features)` returns updated node embeddings of the same shape.
- Use **sum** aggregation via `index_add_`, not mean.
- Include a residual connection and a `LayerNorm`.
- Verify the aggregation against a loop over nodes.

<details><summary><b>Hints</b></summary>

- `edge_index[0]` is the source, `edge_index[1]` the target. `x[source]` gathers one row per edge.
- `torch.zeros_like(x).index_add_(0, target, messages)` scatters messages back to their target nodes, summing collisions.

</details>

**Expected outcome.** Agreement with the loop to floating-point precision, and an output that
keeps the input shape.

In [ ]:
class MyMessagePassing(nn.Module):
    def __init__(self, d_model, edge_dim, hidden=None):
        super().__init__()
        hidden = hidden or 2 * d_model
        self.message = nn.Sequential(
            nn.Linear(2 * d_model + edge_dim, hidden), nn.GELU(),
            nn.Linear(hidden, d_model),
        )
        self.update = nn.Sequential(
            nn.Linear(2 * d_model, hidden), nn.GELU(), nn.Linear(hidden, d_model)
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, edge_index, edge_features):
        source, target = edge_index[0], edge_index[1]
        # TODO 1: one message per edge, from [h_source, h_target, e]
        messages = ____
        # TODO 2: sum the messages arriving at each node
        aggregated = ____
        # TODO 3: update with a residual and normalise
        return ____

set_seed()
layer = MyMessagePassing(d_model=32, edge_dim=len(EDGE_FEATURE_NAMES))
batch = collate_graphs(heldout_graphs[:2])
x = torch.randn(batch.n_nodes, 32)
out = layer(x, batch.edge_index, batch.edge_features)
print(f"nodes {batch.n_nodes}, edges {batch.edge_index.shape[1]} -> "
      f"output {tuple(out.shape)}")

# Verify the aggregation against an explicit loop.
with torch.no_grad():
    src, tgt = batch.edge_index[0], batch.edge_index[1]
    msgs = layer.message(torch.cat([x[src], x[tgt], batch.edge_features], dim=-1))
    vectorised = torch.zeros_like(x).index_add_(0, tgt, msgs)
    looped = torch.zeros_like(x)
    for edge in range(batch.edge_index.shape[1]):
        looped[tgt[edge]] += msgs[edge]
print(f"index_add_ vs explicit loop: "
      f"{float((vectorised - looped).abs().max()):.2e}")

In [ ]:
assert out.shape == x.shape, "a layer must preserve the embedding shape"
assert float((vectorised - looped).abs().max()) < 1e-4, \
    "index_add_ disagrees with the explicit loop"
print("Basic checks passed.")

### Coding Task 10.3 ★★★ — Pretrain on a population of grids

The pretraining task is BERT's, transplanted onto a power system: hide
the state of some buses, reconstruct it from the rest of the network.
Doing that well requires an internal model of how injections, voltages
and topology constrain one another — which is exactly the reusable
knowledge we want.

The training set is several *different grids*, with different sizes and
topologies. A fixed-width model could not even be evaluated on them.

**Your task.** Standardise the features across the population, write the masked
pretraining loop, and train the encoder on all pretraining grids at once.

**Requirements**

- Standardise node and edge features using statistics from the pretraining grids only.
- Mask 15% of buses per batch; score only the masked ones.
- Never mask the slack bus — its voltage is a reference, not an unknown.
- Report the loss curve.

<details><summary><b>Hints</b></summary>

- `MaskedGridModel(node_dim, edge_dim, d_model, n_layers, n_state)` already provides `apply_mask` and a learned `mask_token`.
- The slack flag is `NODE_FEATURE_NAMES.index('is_slack')` in the *unstandardised* features, so build the mask before standardising or keep the flag around.
- Batch with `collate_graphs`, which merges graphs instead of padding them.

</details>

**Expected outcome.** A loss that falls steadily. It will not reach zero — reconstructing a
hidden bus exactly requires solving the power flow, which four rounds of
message passing cannot do.

In [ ]:
N_STATE = len(STATE_FEATURE_NAMES)
SLACK = NODE_FEATURE_NAMES.index("is_slack")

# Standardise across the population, using the pretraining grids only.
all_nodes = torch.cat([g["node_features"] for g in pretrain_graphs])
all_edges = torch.cat([g["edge_features"] for g in pretrain_graphs])
node_mean, node_std = all_nodes.mean(0), all_nodes.std(0).clamp(min=1e-6)
edge_mean, edge_std = all_edges.mean(0), all_edges.std(0).clamp(min=1e-6)

def standardise(graph):
    return {
        "node_features": (graph["node_features"] - node_mean) / node_std,
        "edge_index": graph["edge_index"],
        "edge_features": (graph["edge_features"] - edge_mean) / edge_std,
        "is_slack": graph["node_features"][:, SLACK] > 0.5,
    }

pretrain_std = [standardise(g) for g in pretrain_graphs]
heldout_std = [standardise(g) for g in heldout_graphs]

set_seed()
pretrained = MaskedGridModel(
    node_dim=len(NODE_FEATURE_NAMES), edge_dim=len(EDGE_FEATURE_NAMES),
    d_model=64, n_layers=4, n_state=N_STATE,
)
optimizer = torch.optim.AdamW(pretrained.parameters(), lr=3e-3)
gen = torch.Generator().manual_seed(0)
MASK_RATE, BATCH = 0.15, 8
STEPS = scaled(full=500, fast=60)

pretrain_loss = []
pretrained.train()
for step in range(1, STEPS + 1):
    picks = torch.randint(0, len(pretrain_std), (BATCH,), generator=gen)
    graphs = [pretrain_std[int(i)] for i in picks]
    batch = collate_graphs(graphs)
    is_slack = torch.cat([g["is_slack"] for g in graphs])

    # TODO 1: mask MASK_RATE of the buses, never the slack bus
    mask = ____
    mask = mask & ~is_slack
    if not mask.any():
        continue

    target = batch.node_features[:, :N_STATE]
    # TODO 2: forward with the mask, then score ONLY the masked buses
    reconstruction = ____
    loss = ____

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(pretrained.parameters(), 1.0)
    optimizer.step()
    pretrain_loss.append(float(loss.detach()))
    if step % max(1, STEPS // 6) == 0:
        print(f"step {step:4d}  masked reconstruction loss "
              f"{np.mean(pretrain_loss[-50:]):.4f}")
print(f"\nfirst 50 steps {np.mean(pretrain_loss[:50]):.4f} -> "
      f"last 50 steps {np.mean(pretrain_loss[-50:]):.4f}")

In [ ]:
assert np.mean(pretrain_loss[-50:]) < np.mean(pretrain_loss[:50]), \
    "the pretraining loss did not fall"
assert pretrained.mask_token.shape == (N_STATE,)
_emb = pretrained.embed(collate_graphs(heldout_std[:1]))
assert _emb.shape[0] == heldout_std[0]["node_features"].shape[0], \
    "the encoder must handle a grid size it never saw"
print("Basic checks passed.")

### Coding Task 10.4 ★★★ — Transfer to a grid the model has never seen

The claim being tested: pretraining on *other* grids produces a
representation that makes learning on *this* grid cheaper in labels.

The downstream task is voltage-magnitude estimation from injections and
topology — a state-estimation-flavoured problem. The held-out grid is a
33-bus radial distribution feeder; the pretraining grids are meshed
transmission networks. This is a hard transfer, deliberately.

*Reuses what you wrote in 10.3 — do those first.*

**Your task.** Fine-tune the pretrained encoder and an identically-sized
randomly-initialised one on the held-out grid, across several label
budgets, under a matched optimisation budget.

**Requirements**

- Both conditions get identical data, steps, learning rate and seed.
- At least three label budgets, the smallest being tiny.
- Evaluate on held-out states of the same grid, never on the ones used for fitting.
- Report the relative improvement at each budget.

<details><summary><b>Hints</b></summary>

- Input: hide the voltage magnitude with the pretrained mask token, predict it. Target: the standardised `vm_pu` column.
- Equalise budgets by **steps**, not epochs — otherwise the small-budget condition gets far fewer updates.
- `MaskedGridModel(...)` fresh from `set_seed()` gives you the scratch condition with identical architecture.

</details>

**Expected outcome.** The largest advantage at the smallest budget, shrinking as labels
accumulate. If pretraining loses somewhere, report that too.

In [ ]:
import copy

import pandas as pd

VM = STATE_FEATURE_NAMES.index("vm_pu")
N_TEST = scaled(full=30, fast=6)
BUDGETS = [5, 12] if fast_mode() else [5, 20, 40]
STEPS_FT = scaled(full=150, fast=30)

assert max(BUDGETS) + N_TEST <= len(heldout_std), (
    f"{max(BUDGETS)} training + {N_TEST} test states requested but only "
    f"{len(heldout_std)} sampled"
)

def voltage_task(graphs, mask_token):
    """Hide vm_pu everywhere; the target is the value that was hidden."""
    inputs, targets = [], []
    for g in graphs:
        features = g["node_features"].clone()
        targets.append(features[:, VM].clone())
        features[:, VM] = mask_token[VM]
        inputs.append({**g, "node_features": features})
    return inputs, targets

def run(model, n_train, seed=0):
    torch.manual_seed(seed)
    head = nn.Sequential(nn.Linear(64, 64), nn.GELU(), nn.Linear(64, 1))
    params = list(model.encoder.parameters()) + list(head.parameters())
    optimizer = torch.optim.AdamW(params, lr=1e-3)
    inputs, targets = voltage_task(heldout_std, model.mask_token.detach())
    fit_idx, test_idx = range(n_train), range(len(inputs) - N_TEST, len(inputs))

    model.train()
    for step in range(STEPS_FT):
        pick = [fit_idx[step % n_train]]
        batch = collate_graphs([inputs[i] for i in pick])
        target = torch.cat([targets[i] for i in pick])
        prediction = head(model.encoder(batch)).squeeze(-1)
        loss = torch.nn.functional.mse_loss(prediction, target)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        batch = collate_graphs([inputs[i] for i in test_idx])
        prediction = head(model.encoder(batch)).squeeze(-1)
        truth = torch.cat([targets[i] for i in test_idx])
    # Back to per-unit so the number means something.
    return float((prediction - truth).abs().mean()) * float(node_std[VM])

rows = []
for n_train in BUDGETS:
    # TODO 1: the pretrained condition (deep-copy so budgets are independent)
    warm = copy.deepcopy(pretrained)
    warm_mae = ____

    # TODO 2: the scratch condition, identical architecture and budget
    set_seed()
    cold = ____
    cold.mask_token.data = pretrained.mask_token.data.clone()
    cold_mae = ____

    rows.append({"labels": n_train, "pretrained": warm_mae, "from scratch": cold_mae,
                 "improvement": 1 - warm_mae / cold_mae})
transfer = pd.DataFrame(rows).set_index("labels")
display(transfer.round(4))

In [ ]:
assert set(transfer.columns) == {"pretrained", "from scratch", "improvement"}
assert (transfer[["pretrained", "from scratch"]] > 0).all().all(), \
    "MAE must be positive"
assert (transfer[["pretrained", "from scratch"]] < 1.0).all().all(), \
    "an MAE above 1.0 per unit means something is badly wrong"
print("Basic checks passed.")

### Analysis Task 10.5 ★★★ — Statistical accuracy is not physical validity

A voltage profile can have an excellent MAE and still be a state the
network cannot physically be in. The residual of the AC power balance,

$$S = V \odot \overline{(YV)},$$

recovers the nodal injections a predicted voltage profile implies.
Comparing those with the injections that were actually scheduled tells
you whether the prediction is a realisable operating point or merely a
plausible-looking set of numbers. No statistical metric can answer
that question.

*Reuses what you wrote in 10.3, 10.4 — do those first.*

**Your task.** Train the voltage head properly, confirm it is accurate in per unit,
then compute the power-balance residual of its prediction and compare
it against the feeder's own total load.

**Requirements**

- Train the head first — an untrained model is bad at both statistics and physics, which would demonstrate nothing.
- Report the voltage MAE in per unit, and the mismatch in MW.
- Compare against the residual of the true solved state, which is the numerical floor.
- Express the mismatch as a share of the feeder's total load, because megawatts mean nothing without a denominator.

<details><summary><b>Hints</b></summary>

- `run_power_flow(net)` solves the base case; `internal_state(net)` returns the solved `(vm_pu, va_degree)` in the internal ordering that `get_ybus` uses.
- You predicted magnitudes only, so take the angles from the solved state — and note in your answer that this makes the check *optimistic*.
- `net.load.p_mw.sum()` is the feeder's total demand.

</details>

**Expected outcome.** A voltage MAE well under 1% of nominal — by any statistical standard a
good result — alongside a power mismatch that is a large fraction of
everything the feeder consumes. That gap is the entire point of the
task.

In [ ]:
from ai_power_course.grid.physics import internal_state, power_balance_residual

net = load_network(HELDOUT)
assert run_power_flow(net), "the base case must converge"
vm_true, va_true = internal_state(net)
base_graph = standardise(net_to_graph(net))

# Train the head. An untrained one would be bad at BOTH statistics and
# physics, and would prove nothing; we want a model that looks good.
set_seed()
probe_model = copy.deepcopy(pretrained)
head_check = nn.Sequential(nn.Linear(64, 64), nn.GELU(), nn.Linear(64, 1))
optimizer = torch.optim.AdamW(
    list(probe_model.encoder.parameters()) + list(head_check.parameters()),
    lr=1e-3,
)
fit_inputs, fit_targets = voltage_task(
    heldout_std[:40], pretrained.mask_token.detach())

probe_model.train()
for step in range(scaled(full=400, fast=120)):
    batch = collate_graphs([fit_inputs[step % len(fit_inputs)]])
    target = fit_targets[step % len(fit_targets)]
    # TODO 1: forward through encoder and head, MSE loss, optimise
    prediction = ____
    loss = ____
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

inputs, _truths = voltage_task([base_graph], pretrained.mask_token.detach())
probe_model.eval()
with torch.no_grad():
    predicted_std = head_check(
        probe_model.encoder(collate_graphs(inputs))).squeeze(-1)
vm_pred = (predicted_std * node_std[VM] + node_mean[VM]).numpy()

# TODO 2: statistical error, in per unit
voltage_mae = ____
print(f"voltage MAE: {voltage_mae:.4f} per unit "
      f"({voltage_mae * 100:.2f}% of nominal)")
print(f"predicted range {vm_pred.min():.3f}-{vm_pred.max():.3f} pu, "
      f"true {vm_true.min():.3f}-{vm_true.max():.3f} pu")

# TODO 3: the residual of the PREDICTED state, and of the true one
predicted_residual = ____
true_residual = ____
total_load = float(net.load.p_mw.sum())
print(f"\npredicted state: {predicted_residual['p_mismatch_mean_mw']:7.3f} MW "
      f"mean, {predicted_residual['p_mismatch_max_mw']:7.3f} MW max")
print(f"solved state   : {true_residual['p_mismatch_mean_mw']:7.3f} MW "
      f"mean, {true_residual['p_mismatch_max_mw']:7.3f} MW max")
print(f"\ntotal feeder load: {total_load:.2f} MW")
share = predicted_residual["p_mismatch_mean_mw"] / total_load
print(f"mean mismatch as a share of it: {share:.1%}")

In [ ]:
assert true_residual["p_mismatch_max_mw"] < 1e-3, \
    "the solved state should satisfy its own equations"
if fast_mode():
    print("Reduced configuration: skipping the accuracy check — 120 steps")
    print("is not enough to train the head to the point the task is making.")
else:
    assert voltage_mae < 0.02, \
        "train the head further — the point needs an ACCURATE model"
assert predicted_residual["p_mismatch_mean_mw"] > \
    true_residual["p_mismatch_mean_mw"], \
    "a prediction cannot beat the exact solution on its own residual"
print("Basic checks passed.")

### Reflection Question 10.6 ★★★ — Design your own grid foundation model

The open question this course has been building towards:

> **What should the tokens of an electrical grid foundation model be?**

A language model has words. An image model has patches. A grid has …
buses? Lines? Substations? Time-steps of a bus? Feeders? Whole snapshots?

Nobody knows. The models that exist — GridFM, PowerGraph, the
grid-specific variants appearing since 2024 — disagree with each other,
and the disagreement is not a detail: the tokenisation decides what can
be masked, what transfers, and what the model can generalise across.

This is a genuinely open research question and there is no answer key.

**Your task.** Design a foundation model for a power-systems problem you actually care
about. Fill in the template below. You are being assessed on whether the
pieces are *consistent with each other* — pretraining objective with
token choice, evaluation with claim — not on ambition.

**Requirements**

- Name the token explicitly and say what one token *is*.
- The pretraining objective must be self-supervised: no labels.
- Say what you would evaluate zero-shot and what needs fine-tuning.
- Include at least one *physical* validation, separate from the statistical one.
- Name the failure mode you consider most likely, and the experiment that would reveal it.

<details><summary><b>Hints</b></summary>

- Start from what you want to transfer ACROSS — grids? seasons? voltage levels? — because that determines what a token must be comparable across.
- Per-unit normalisation is what makes a 110 kV and a 20 kV measurement commensurable. Without it, your tokens are not comparable and nothing transfers.
- A good sanity check: would your masked-reconstruction task be trivially solvable by interpolation? If so it will teach the model nothing.

</details>

**Expected outcome.** Internal consistency. The most common failure is a token choice that
cannot support the claimed downstream task — for example tokenising
whole snapshots and then claiming per-bus anomaly localisation.

### My Grid Foundation Model

**1. The problem I care about.**
*(Two sentences. What decision would this support?)*

**2. The token.**
*(What is one token? A bus at one instant? A line? A substation-day? Be
specific about its dimensionality and units.)*

**3. What makes tokens comparable across grids.**
*(Per-unit? Normalised by rating? By local mean? Why does this work?)*

**4. The pretraining corpus.**
*(What data, how much, from where? Is it obtainable?)*

**5. The self-supervised objective.**
*(Masked reconstruction of what? Next-what prediction? Contrastive
between which pairs?)*

**6. Architecture, and the symmetry it builds in.**
*(What is invariant or equivariant, and why does that match the domain?)*

**7. Downstream tasks.**
| Task | Zero-shot, probe, or fine-tune? | Why |
| --- | --- | --- |
| … | … | … |

**8. Evaluation.**
- Statistical: …
- Physical: …
- Baseline it must beat: …

**9. The most likely failure mode, and the experiment that reveals it.**
*(Be specific. "It might not work" is not a failure mode.)*

**10. The thing I am least sure about.**

---

## Finished?

Some things worth doing before you look at the solutions.

1. **Re-read your reflection answers.** They are the ones that are hard to
   mark yourself and the ones most worth discussing with a supervisor or a
   colleague.
2. **Find a task where your number disagrees with a neighbour's** and work
   out which of you changed something. Seeds are fixed throughout; a
   difference always has a cause.
3. **Then** open `solution.ipynb`, and compare *reasoning*, not just
   output. Several tasks have more than one defensible answer.

The last task has no solution in any meaningful sense. What the tokens of a
grid foundation model should be is an open research question, and the
worked example in the solution notebook is one design among many.